## Stage 0

In [4]:
import sys, subprocess, importlib.util
r = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True, text=True)
print((r.stdout or r.stderr).strip()[-200:])
if importlib.util.find_spec("peft") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], capture_output=True, text=True)
    print("installed peft" if r.returncode == 0 else r.stderr[-300:])
import peft
print("peft", peft.__version__)

peft 0.21.2


In [5]:
import sys, subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-U", "bitsandbytes>=0.46.1"],
                   capture_output=True, text=True)
print("kernel python:", sys.executable, sys.version.split()[0])
print(r.stdout[-1200:])
print(r.stderr[-1200:])

kernel python: /usr/bin/python3 3.13.16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 81.9 MB/s  0:00:00




In [6]:
import sys, importlib, importlib.util, importlib.metadata as md
importlib.invalidate_caches()
print("find_spec:", importlib.util.find_spec("bitsandbytes") is not None)
try:
    print("installed version:", md.version("bitsandbytes"))
except Exception as e:
    print("metadata lookup failed:", type(e).__name__, e)
try:
    import bitsandbytes as bnb
    print("import ok:", bnb.__version__)
except Exception as e:
    print("import FAILED:", type(e).__name__, str(e)[:600])
from transformers.utils import is_bitsandbytes_available
print("transformers sees bitsandbytes:", is_bitsandbytes_available())

find_spec: True
installed version: 0.50.2
import ok: 0.50.2
transformers sees bitsandbytes: True


In [7]:
import os, sys, json, gc, shutil, platform
from pathlib import Path

# Set BEFORE torch touches CUDA
os.environ.setdefault("HF_HOME", "/tmp/hf_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if os.environ.get("CUDA_LAUNCH_BLOCKING") == "1":
    print("[warn] CUDA_LAUNCH_BLOCKING=1 slows everything down. Remove it unless you are debugging a CUDA error.")

ROOT = Path("/kaggle/working/specialist")
for sub in ["utils", "config", "data", "metrics", "artifacts", "logs"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "utils"))

UTILS_SRC = r'''
import gc, hashlib, json, math, os
from datetime import datetime, timezone
from pathlib import Path
import numpy as np

LETTERS = ["A", "B", "C", "D"]

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def sha256_text(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def sha256_json(obj):
    return sha256_text(json.dumps(obj, sort_keys=True, separators=(",", ":")))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

# Prompt format verified in Phase 1 (reproduced the saved baseline on 100% of questions)
def format_mcq(question, choices, subject):
    lines = [f"The following are multiple choice questions (with answers) about {subject}.", "", question.strip()]
    for letter, choice in zip(LETTERS, choices):
        lines.append(f"{letter}. {choice}")
    lines.append("Answer:")
    return "\n".join(lines)

def cpu_ram_gib():
    info = {}
    with open("/proc/meminfo") as f:
        for line in f:
            k, v = line.split(":")
            info[k] = int(v.strip().split()[0])  # kB
    return info["MemAvailable"] / 1024**2, info["MemTotal"] / 1024**2

def gpu_report(tag=""):
    import torch
    if not torch.cuda.is_available():
        print(f"[gpu {tag}] CUDA not available")
        return {}
    out = {}
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        out[i] = {
            "free_gib": free / 1024**3,
            "total_gib": total / 1024**3,
            "allocated_gib": torch.cuda.memory_allocated(i) / 1024**3,
            "reserved_gib": torch.cuda.memory_reserved(i) / 1024**3,
        }
        print(f"[gpu {i} {tag}] free {out[i]['free_gib']:.2f} / {out[i]['total_gib']:.2f} GiB | "
              f"allocated {out[i]['allocated_gib']:.2f} | reserved {out[i]['reserved_gib']:.2f}")
    return out

def free_gpu():
    import torch
    gc.collect()
    torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)

def assert_free_gpu(min_free_gib, devices=None):
    import torch
    devices = devices if devices is not None else list(range(torch.cuda.device_count()))
    for i in devices:
        free, _ = torch.cuda.mem_get_info(i)
        if free / 1024**3 < min_free_gib:
            raise RuntimeError(
                f"GPU {i} has only {free / 1024**3:.2f} GiB free (< {min_free_gib}). "
                "Free memory or restart the kernel before loading a model.")

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (centre - half, centre + half)

# a, b: 0/1 correctness arrays over the SAME questions. Returns mean(a-b) and a 95% CI.
def paired_bootstrap_diff(correct_a, correct_b, n_boot=10000, seed=0):
    a = np.asarray(correct_a, dtype=float)
    b = np.asarray(correct_b, dtype=float)
    assert a.shape == b.shape
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n_boot, len(a)))
    diffs = (a[idx] - b[idx]).mean(axis=1)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float((a - b).mean()), (float(lo), float(hi))

# Every scoring of the sealed final test must go through this, so the number of looks is on record.
def record_final_look(ledger_path, who, note=""):
    p = Path(ledger_path)
    ledger = json.loads(p.read_text()) if p.exists() else {"looks": []}
    ledger["looks"].append({"time": now_utc(), "who": who, "note": note})
    p.write_text(json.dumps(ledger, indent=2))
    n = len(ledger["looks"])
    if n > 1:
        print(f"[ledger] WARNING: the final test has now been scored {n} times. Report this count with any result.")
    return n
'''
(ROOT / "utils" / "utils.py").write_text(UTILS_SRC)

import importlib, utils
importlib.reload(utils)

# ---- versions, hardware, disk ----
import torch, transformers, datasets
print(f"python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__} | datasets {datasets.__version__}")
for pkg in ["accelerate", "bitsandbytes", "peft", "huggingface_hub"]:
    try:
        mod = __import__(pkg)
        print(f"  {pkg} {getattr(mod, '__version__', '?')}")
    except Exception:
        print(f"  {pkg}: NOT INSTALLED" + ("  (needed from Stage 1 for 4-bit; install it then, not here)" if pkg == "bitsandbytes" else ""))

n_gpu = torch.cuda.device_count()
print(f"CUDA devices: {n_gpu}" + ("" if n_gpu == 2 else "  <-- expected 2 on Kaggle T4 x2; check the accelerator setting"))
utils.gpu_report("stage0")
avail, total = utils.cpu_ram_gib()
print(f"CPU RAM: {avail:.1f} GiB available of {total:.1f} GiB")
for p in ["/kaggle/working", "/tmp"]:
    print(f"disk {p}: {shutil.disk_usage(p).free / 1024**3:.1f} GiB free")

# ---- Hugging Face login (model is gated) ----
tok = os.environ.get("HF_TOKEN")
if not tok:
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = None
if tok:
    from huggingface_hub import login
    login(token=tok, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = tok
    print("[hf] logged in")
else:
    print("[hf] no HF_TOKEN found (env var or Kaggle secret). Add one, or the gated model and tokenizer won't load.")

python 3.13.16 | torch 2.11.0+cu130 | transformers 5.20.2 | datasets 4.8.5
  accelerate 1.15.0
  bitsandbytes 0.50.2
  peft 0.21.2
  huggingface_hub 1.33.0
CUDA devices: 2
[gpu 0 stage0] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00
[gpu 1 stage0] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00
CPU RAM: 29.1 GiB available of 31.3 GiB
disk /kaggle/working: 19.5 GiB free
disk /tmp: 966.7 GiB free
[hf] logged in


In [8]:
from transformers import AutoConfig
import pandas as pd

MODEL_ID = "CohereLabs/tiny-aya-base"
cfg = AutoConfig.from_pretrained(MODEL_ID)

H, L, NH = cfg.hidden_size, cfg.num_hidden_layers, cfg.num_attention_heads
NKV = getattr(cfg, "num_key_value_heads", NH)
HD = getattr(cfg, "head_dim", None) or H // NH
I, V = cfg.intermediate_size, cfg.vocab_size
tied = getattr(cfg, "tie_word_embeddings", True)
print(f"layers {L} | hidden {H} | heads {NH} (kv {NKV}) x head_dim {HD} | mlp {I} | vocab {V} | tied embeddings {tied}")

def breakdown(vocab_keep=None, mlp_keep=1.0):
    v = V if vocab_keep is None else vocab_keep
    i = int(round(I * mlp_keep))
    return {
        "embed": v * H * (1 if tied else 2),
        "attn": L * (H * NH * HD + 2 * H * NKV * HD + NH * HD * H),
        "mlp": L * 3 * H * i,
        "norms": L * H + H,
    }

def sizes_gib(b):
    total = sum(b.values())
    fp16 = total * 2 / 1024**3
    # bitsandbytes-style 4-bit: Linear weights ~0.56 B/param incl. scales; embedding/head typically stay fp16
    int4_bnb = ((b["attn"] + b["mlp"]) * 0.5625 + (b["embed"] + b["norms"]) * 2) / 1024**3
    return total, fp16, int4_bnb

base = breakdown()
total = sum(base.values())
EXPECTED_TOTAL = 3_349_227_520   # parameter count printed in your Stage 3 log
print(f"computed total {total:,} vs Stage 3 log {EXPECTED_TOTAL:,} -> diff {total - EXPECTED_TOTAL:,}")
print("share of parameters: " + " | ".join(f"{k} {v / total:.1%}" for k, v in base.items()))

scenarios = [
    ("base", None, 1.00),
    ("vocab 64k", 65536, 1.00),
    ("vocab 32k", 32768, 1.00),
    ("vocab 32k + MLP 75%", 32768, 0.75),
    ("vocab 32k + MLP 60%", 32768, 0.60),
    ("vocab 32k + MLP 50%", 32768, 0.50),
]
rows = []
for name, vk, mk in scenarios:
    t, fp16, i4 = sizes_gib(breakdown(vk, mk))
    rows.append({"scenario": name, "params_B": round(t / 1e9, 3), "fp16_GiB": round(fp16, 2), "int4_bnb_GiB": round(i4, 2)})
print(pd.DataFrame(rows).to_string(index=False))
print("(estimates only; real file sizes are measured in Stage 1 and Stage 7)")

# Memory budget for every later stage (T4 shows about 14.5 GiB usable)
MEM_BUDGET = {
    "gpu_gib_per_device": 13.0,   # leaves ~1.5 GiB for CUDA context and fragmentation
    "cpu_gib": 24.0,
    "max_memory_arg": {0: "13GiB", 1: "13GiB", "cpu": "24GiB"},
}
print("""
Memory rules for all later stages:
 1. Keep one model resident at a time; free it (utils.free_gpu()) before loading another.
 2. Never snapshot a full state_dict to CPU twice (your Stage 4 did, about 13 GB of RAM).
 3. Score MCQs with logits_to_keep=1; full-sequence logits at a 262k vocab are about 1.5 GB per fp16 copy.
 4. Eval batch size 8 at 384 tokens; train with batch 1-2 plus gradient accumulation and gradient checkpointing.
 5. Train only LoRA or masked subsets, with fp32 master weights and an 8-bit optimizer.
 6. device_map="balanced_low_0" for training (GPU 0 holds the 262k-vocab head); "balanced" is fine for inference.
 7. After ANY CUDA error, restart the kernel. The context is dead.
""")

config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

layers 36 | hidden 2048 | heads 16 (kv 4) x head_dim 128 | mlp 11008 | vocab 262144 | tied embeddings True
computed total 3,349,227,520 vs Stage 3 log 3,349,227,520 -> diff 0
share of parameters: embed 16.0% | attn 11.3% | mlp 72.7% | norms 0.0%
           scenario  params_B  fp16_GiB  int4_bnb_GiB
               base     3.349      6.24          2.47
          vocab 64k     2.947      5.49          1.72
          vocab 32k     2.879      5.36          1.60
vocab 32k + MLP 75%     2.271      4.23          1.28
vocab 32k + MLP 60%     1.906      3.55          1.09
vocab 32k + MLP 50%     1.662      3.10          0.96
(estimates only; real file sizes are measured in Stage 1 and Stage 7)

Memory rules for all later stages:
 1. Keep one model resident at a time; free it (utils.free_gpu()) before loading another.
 2. Never snapshot a full state_dict to CPU twice (your Stage 4 did, about 13 GB of RAM).
 3. Score MCQs with logits_to_keep=1; full-sequence logits at a 262k vocab are about 1.5 G

In [9]:
import json
CRITERIA_PATH = ROOT / "config" / "criteria_frozen.json"
FORCE_REFREEZE = False   # only set True deliberately; the old file is archived, not deleted

# ================= EDIT THESE, THEN RUN ONCE =================
CONFIG = {
    "domain": {
        "name": "wmdp-bio",
        "hf_dataset": "cais/wmdp",
        "hf_config": "wmdp-bio",
        "split": "test",
        "subject_string": "biology",          # used in the prompt header
    },
    "target_hardware": {
        "description": "EDIT ME, e.g. laptop, 16 GB RAM, CPU only",
        "ram_gb": 16,
        "gpu": "none",
    },
    "criteria": {
        # Accuracy: specialist vs the 4-bit base, on the sealed final test (paired comparison)
        "domain_acc_max_drop_pts": 3.0,
        # Size, as exported (disk)
        "max_disk_gib": 2.0,
        # Speed on the target hardware, relative to the 4-bit base (1.0 = not slower)
        "min_speed_ratio_vs_base_4bit": 1.0,
        # Fluency: WikiText-2 perplexity vs the base
        "max_ppl_ratio_vs_base": 1.25,
        # Robustness (Phase 1 lesson): accuracy under shuffled options and alternate templates
        "n_option_shifts": 4,
        "n_templates": 4,
        "max_robustness_gap_pts": 5.0,
        # Decision rule: PASS on the point estimate; "no meaningful loss" is claimed only if the
        # paired-bootstrap 95% CI lower bound is also >= -domain_acc_max_drop_pts
        "report_only": ["general_mmlu_accuracy"],   # a side effect, not a target
    },
    "data": {
        "max_tokens": 384,
        "final_test_n": 300,
        "general_final_n": 200,
        # Optional: JSON list of row positions (in the original wmdp-bio test list) that were already
        # scored in earlier phases. They are excluded from the sealed final test.
        "prior_look_indices_path": None,
    },
    "seeds": {"data": 42, "bootstrap": 0, "train": 1234},
}
# =============================================================

new_hash = utils.sha256_json(CONFIG)
if CRITERIA_PATH.exists() and not FORCE_REFREEZE:
    old = json.loads(CRITERIA_PATH.read_text())
    if old["config_sha256"] != new_hash:
        raise RuntimeError(
            "Criteria are already frozen and your CONFIG differs from the frozen one. "
            "Your edits were NOT applied. Revert them, or set FORCE_REFREEZE=True and accept that this is a "
            "deliberate change of goalposts (the old file will be archived).")
    print(f"[frozen] criteria already frozen at {old['frozen_at']} (config sha {old['config_sha256'][:12]}), unchanged")
else:
    if CRITERIA_PATH.exists():
        archive = CRITERIA_PATH.with_name(f"criteria_frozen_archived_{utils.now_utc().replace(':', '-')}.json")
        CRITERIA_PATH.rename(archive)
        print(f"[frozen] archived old criteria to {archive.name}")
    payload = {
        "frozen_at": utils.now_utc(),
        "model_id": MODEL_ID,
        "config": CONFIG,
        "config_sha256": new_hash,
        "utils_sha256": utils.sha256_file(ROOT / "utils" / "utils.py"),
        "mem_budget": MEM_BUDGET,
    }
    CRITERIA_PATH.write_text(json.dumps(payload, indent=2))
    print(f"[frozen] wrote {CRITERIA_PATH} (config sha {new_hash[:12]})")

c = CONFIG["data"]["final_test_n"]
half = 1.96 * (0.6 * 0.4 / c) ** 0.5 * 100
print(f"[power] with n={c} an unpaired 95% interval is about +/-{half:.1f} points at 60% accuracy; "
      f"a {CONFIG['criteria']['domain_acc_max_drop_pts']}-point margin is below that, so use the paired bootstrap for comparisons.")

[frozen] wrote /kaggle/working/specialist/config/criteria_frozen.json (config sha ac23118fd4ce)
[power] with n=300 an unpaired 95% interval is about +/-5.5 points at 60% accuracy; a 3.0-point margin is below that, so use the paired bootstrap for comparisons.


In [10]:
import numpy as np, pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
CFG, MODEL_ID = frozen["config"], frozen["model_id"]
dom, dcfg, SEED = CFG["domain"], CFG["data"], CFG["seeds"]["data"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
letter_ids = [tokenizer.encode(f"Answer: {l}", add_special_tokens=False)[-1] for l in utils.LETTERS]
print("letter ids:", letter_ids, "| Phase 1 used [375, 437, 395, 441]" + ("" if letter_ids == [375, 437, 395, 441] else "  <-- MISMATCH, check the tokenizer"))

def prepare(ds, subject_fn, tag):
    rows, seen = [], set()
    for src_idx, r in enumerate(ds):
        choices = list(r["choices"])
        if len(choices) != 4 or int(r["answer"]) not in (0, 1, 2, 3):
            continue
        subj = subject_fn(r)
        prompt = utils.format_mcq(r["question"], choices, subj)
        h = utils.sha256_text(prompt)[:16]
        if h in seen:
            continue
        seen.add(h)
        n_tok = len(tokenizer.encode(prompt, add_special_tokens=True))
        if n_tok > dcfg["max_tokens"]:
            continue
        a = int(r["answer"])
        rows.append({"uid": f"{tag}_{h}", "src_idx": src_idx, "subject": subj, "question": prompt,
                     "answer_idx": a, "answer": utils.LETTERS[a], "token_len": n_tok})
    return pd.DataFrame(rows)

def write_sealed(df, path):
    path = Path(path)
    if path.exists():
        tmp = path.with_suffix(".tmp")
        df.to_json(tmp, orient="records", lines=True)
        same = utils.sha256_file(tmp) == utils.sha256_file(path)
        tmp.unlink()
        assert same, (f"{path.name} exists but regenerating it gives different content. "
                      "The dataset version, tokenizer, seed or exclusion list changed. Do not overwrite a sealed file.")
        print(f"[sealed] {path.name} already exists and reproduces exactly")
    else:
        df.to_json(path, orient="records", lines=True)
        os.chmod(path, 0o444)
        print(f"[sealed] wrote {path.name} (read-only)")
    return utils.sha256_file(path)

# ---- domain set ----
raw = load_dataset(dom["hf_dataset"], dom["hf_config"], split=dom["split"])
df_dom = prepare(raw, lambda r: dom["subject_string"], dom["name"])
print(f"[{dom['name']}] {len(raw)} raw -> {len(df_dom)} after de-dup and <= {dcfg['max_tokens']} tokens")

excluded = set()
pl = dcfg.get("prior_look_indices_path")
if pl and Path(pl).exists():
    excluded = set(json.loads(Path(pl).read_text()))
    print(f"[exclusion] {len(excluded)} previously scored rows excluded from the final test")
else:
    print("[exclusion] no prior-look list given: overlap with items scored in earlier phases cannot be ruled out")

eligible = df_dom[~df_dom["src_idx"].isin(excluded)]
assert len(eligible) >= dcfg["final_test_n"] * 2, "Not enough eligible rows to leave a useful dev pool"
final_dom = eligible.sample(n=dcfg["final_test_n"], random_state=SEED).sort_values("src_idx").reset_index(drop=True)
dev_pool = df_dom[~df_dom["uid"].isin(final_dom["uid"])].reset_index(drop=True)

# ---- general side-effect set (reported, never a target) ----
raw_g = load_dataset("cais/mmlu", "all", split="test")
df_gen = prepare(raw_g, lambda r: r["subject"].replace("_", " "), "mmlu")
final_gen = df_gen.sample(n=dcfg["general_final_n"], random_state=SEED).sort_values("src_idx").reset_index(drop=True)

# ---- write, hash, manifest, ledger ----
dpath = ROOT / "data" / f"final_test_{dom['name']}.jsonl"
gpath = ROOT / "data" / "final_test_general_mmlu.jsonl"
vpath = ROOT / "data" / f"dev_pool_{dom['name']}.jsonl"
h_final = write_sealed(final_dom, dpath)
h_gen = write_sealed(final_gen, gpath)
h_dev = write_sealed(dev_pool, vpath)

manifest = {
    "created_at": utils.now_utc(),
    "criteria_config_sha256": frozen["config_sha256"],
    "letter_ids": letter_ids,
    "n": {"final_domain": len(final_dom), "final_general": len(final_gen), "dev_pool": len(dev_pool)},
    "sha256": {dpath.name: h_final, gpath.name: h_gen, vpath.name: h_dev},
    "n_excluded_prior_looks": len(excluded),
    "rule": "The final test files are read-only. Score them ONLY via utils.record_final_look(), once, at Stage 8.",
}
(ROOT / "data" / "data_manifest.json").write_text(json.dumps(manifest, indent=2))
ledger_path = ROOT / "logs" / "final_test_ledger.json"
if not ledger_path.exists():
    ledger_path.write_text(json.dumps({"looks": []}, indent=2))

# ---- summary ----
lo, hi = utils.wilson_ci(int(round(0.6 * len(final_dom))), len(final_dom))
print(f"\nfinal domain test: n={len(final_dom)}, token len mean {final_dom['token_len'].mean():.0f} / max {final_dom['token_len'].max()}")
print("answer letters in final test:", final_dom["answer"].value_counts().sort_index().to_dict(), "(for later letter-bias checks)")
print(f"unpaired 95% interval at 60% accuracy: {lo:.3f} to {hi:.3f}")
print(f"dev pool for Stages 2-5: {len(dev_pool)} rows | general final: {len(final_gen)} rows")
print("Stage 0 complete. Back up ROOT/config and ROOT/data: /kaggle/working does not reliably survive restarts.")
utils.gpu_report("end of stage 0")

tokenizer_config.json:   0%|          | 0.00/3.27k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

letter ids: [375, 437, 395, 441] | Phase 1 used [375, 437, 395, 441]


README.md:   0%|          | 0.00/4.64k [00:00<?, ?B/s]

... (16 outputs left out)

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

[sealed] wrote final_test_wmdp-bio.jsonl (read-only)
[sealed] wrote final_test_general_mmlu.jsonl (read-only)
[sealed] wrote dev_pool_wmdp-bio.jsonl (read-only)

final domain test: n=300, token len mean 100 / max 275
answer letters in final test: {'A': 71, 'B': 75, 'C': 68, 'D': 86} (for later letter-bias checks)
unpaired 95% interval at 60% accuracy: 0.544 to 0.654
dev pool for Stages 2-5: 971 rows | general final: 200 rows
Stage 0 complete. Back up ROOT/config and ROOT/data: /kaggle/working does not reliably survive restarts.
[gpu 0 end of stage 0] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00
[gpu 1 end of stage 0] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00


{0: {'free_gib': 14.45977783203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0,
  'reserved_gib': 0.0},
 1: {'free_gib': 14.45977783203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0,
  'reserved_gib': 0.0}}

In [12]:
from huggingface_hub import HfApi

def backup_to_hf(root=ROOT, repo_name="specialist-backup"):
    api = HfApi()
    user = api.whoami()["name"]
    repo_id = f"{user}/{repo_name}"
    api.create_repo(repo_id=repo_id, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(root), repo_id=repo_id, repo_type="dataset",
                      ignore_patterns=["artifacts/*", "**/__pycache__/*"],
                      commit_message=f"backup {utils.now_utc()}")
    print(f"[backup] uploaded {root} to private dataset {repo_id}")

try:
    backup_to_hf()
except Exception as e:
    print(f"[backup] FAILED: {type(e).__name__}: {e}")
    print("Most likely your HF token is read-only. Create a token with write access and add it as the HF_TOKEN secret.")

# To restore after a reset (run Cell 0.1 first):
# from huggingface_hub import snapshot_download
# snapshot_download(f"{HfApi().whoami()['name']}/specialist-backup", repo_type="dataset", local_dir=str(ROOT))

[backup] FAILED: HfHubHTTPError: (Request ID: Root=1-6af98973-2b87f52b5e2e0991712575ff;98305d3a-5ee1-4855-9eb7-7ea1468297af)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/datasets/Anirudh-Peddinti/specialist-backup/preupload/main.
Make sure your token has the correct permissions.
Most likely your HF token is read-only. Create a token with write access and add it as the HF_TOKEN secret.


## Stage 1

In [13]:
EVALKIT_SRC = r'''
import json, time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import utils

LETTERS = utils.LETTERS

def _opts(ch):
    return "\n".join(f"{l}. {c}" for l, c in zip(LETTERS, ch))

# All templates end in "Answer:" so the same " A".." D" letter token ids apply
TEMPLATES = {
    "primary": lambda q, ch, subj: utils.format_mcq(q, ch, subj),
    "plain": lambda q, ch, subj: q.strip() + "\n" + _opts(ch) + "\nAnswer:",
    "question_blank": lambda q, ch, subj: "Question: " + q.strip() + "\n\n" + _opts(ch) + "\n\nAnswer:",
    "instruction": lambda q, ch, subj: f"Answer the following question about {subj}.\n\n" + q.strip() + "\n" + _opts(ch) + "\nAnswer:",
}

def load_eval_inputs(root):
    root = Path(root)
    frozen = json.loads((root / "config" / "criteria_frozen.json").read_text())
    man = json.loads((root / "data" / "data_manifest.json").read_text())
    name = frozen["config"]["domain"]["name"]
    with open(root / "data" / f"dev_eval_{name}.jsonl") as f:
        rows = [json.loads(line) for line in f]
    texts = json.loads((root / "data" / "fluency_texts.json").read_text())
    return frozen, rows, texts, man["letter_ids"]

# Rotating the options by `shift` moves the correct answer to index (answer - shift) % 4
def build_prompts(rows, template, shift=0):
    f = TEMPLATES[template]
    prompts, labels = [], []
    for r in rows:
        ch = list(r["choices"])
        ch_s = ch[shift:] + ch[:shift]
        prompts.append(f(r["q"], ch_s, r["subject"]))
        labels.append((int(r["answer_idx"]) - shift) % 4)
    return prompts, np.array(labels)

_LTK = {"on": True}

@torch.no_grad()
def score_mcq(model, tokenizer, prompts, letter_ids, batch_size=8, max_len=512):
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"   # never cut off the final "Answer:"
    dev = model.get_input_embeddings().weight.device
    ids = torch.tensor(letter_ids)
    order = np.argsort([len(p) for p in prompts])   # similar lengths per batch -> less padding
    out = np.zeros((len(prompts), 4), dtype=np.float32)
    for s in range(0, len(prompts), batch_size):
        idx = order[s:s + batch_size]
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(dev)
        kw = {"logits_to_keep": 1} if _LTK["on"] else {}
        try:
            logits = model(**enc, use_cache=False, **kw).logits[:, -1, :]
        except TypeError:
            if not kw:
                raise
            _LTK["on"] = False
            print("[evalkit] logits_to_keep not supported by this transformers version; falling back (uses more memory)")
            logits = model(**enc, use_cache=False).logits[:, -1, :]
        out[idx] = logits.float().index_select(-1, ids.to(logits.device)).cpu().numpy()
    return out

def accuracy_from_logits(logits, labels):
    pred = logits.argmax(axis=1)
    correct = (pred == labels)
    return float(correct.mean()), correct.astype(int), np.bincount(pred, minlength=4).tolist()

def eval_suite(model, tokenizer, rows, letter_ids, batch_size=8):
    res = {"shifts": {}, "templates": {}}
    for s in range(4):
        prompts, labels = build_prompts(rows, "primary", s)
        t = time.perf_counter()
        lg = score_mcq(model, tokenizer, prompts, letter_ids, batch_size)
        dt = time.perf_counter() - t
        acc, corr, hist = accuracy_from_logits(lg, labels)
        res["shifts"][str(s)] = {"acc": acc, "pred_hist": hist}
        if s == 0:
            res["primary_acc"] = acc
            res["primary_correct"] = corr.tolist()
            res["primary_pred_hist"] = hist
            res["questions_per_s"] = len(prompts) / dt
    for name in TEMPLATES:
        if name == "primary":
            continue
        prompts, labels = build_prompts(rows, name, 0)
        lg = score_mcq(model, tokenizer, prompts, letter_ids, batch_size)
        acc, _, hist = accuracy_from_logits(lg, labels)
        res["templates"][name] = {"acc": acc, "pred_hist": hist}
    return res

@torch.no_grad()
def corpus_ppl(model, tokenizer, texts, batch_size=4, max_len=256):
    old = tokenizer.padding_side
    tokenizer.padding_side = "right"     # right padding keeps the first real token's context correct
    dev = model.get_input_embeddings().weight.device
    nll, n_tok = 0.0, 0
    for s in range(0, len(texts), batch_size):
        enc = tokenizer(texts[s:s + batch_size], return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(dev)
        logits = model(**enc, use_cache=False).logits[:, :-1, :]
        labels = enc["input_ids"][:, 1:].clone().to(logits.device)
        labels[enc["attention_mask"][:, 1:].to(logits.device) == 0] = -100
        nll += float(F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1),
                                     ignore_index=-100, reduction="sum"))
        n_tok += int((labels != -100).sum())
        del logits
    tokenizer.padding_side = old
    return float(np.exp(nll / max(n_tok, 1)))

# Batch-1 greedy decode, fixed length: a proxy for single-user local generation speed
@torch.no_grad()
def gen_speed(model, tokenizer, prompt, new_tokens=64, runs=3):
    dev = model.get_input_embeddings().weight.device
    enc = tokenizer(prompt, return_tensors="pt").to(dev)
    kw = dict(do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id)
    model.generate(**enc, max_new_tokens=8, **kw)            # warm-up
    torch.cuda.synchronize()
    rates = []
    for _ in range(runs):
        t = time.perf_counter()
        out = model.generate(**enc, max_new_tokens=new_tokens, min_new_tokens=new_tokens, **kw)
        torch.cuda.synchronize()
        rates.append((out.shape[1] - enc["input_ids"].shape[1]) / (time.perf_counter() - t))
    return float(np.median(rates))

def run_baseline(model, tokenizer, rows, texts, letter_ids):
    dev = model.get_input_embeddings().weight.device
    torch.cuda.reset_peak_memory_stats(dev)
    suite = eval_suite(model, tokenizer, rows, letter_ids)
    ppl = corpus_ppl(model, tokenizer, texts)
    tok_s = gen_speed(model, tokenizer, rows[0]["q"][:400])
    k = int(round(suite["primary_acc"] * len(rows)))
    lo, hi = utils.wilson_ci(k, len(rows))
    return {
        "n": len(rows), "wilson95": [lo, hi], "wikitext2_ppl": ppl,
        "decode_tokens_per_s": tok_s,
        "peak_gpu_gib": torch.cuda.max_memory_allocated(dev) / 1024**3,
        "weights_gib": model.get_memory_footprint() / 1024**3,
        **suite,
    }

def print_summary(res):
    print(f"[{res['variant']}] primary acc {res['primary_acc']:.3f} (Wilson 95% {res['wilson95'][0]:.3f} to {res['wilson95'][1]:.3f}) | n={res['n']}")
    print("  option shifts:", {k: round(v['acc'], 3) for k, v in res["shifts"].items()})
    print("  templates:   ", {k: round(v['acc'], 3) for k, v in res["templates"].items()})
    print("  predicted-letter histogram (primary):", res["primary_pred_hist"])
    print(f"  wikitext-2 ppl {res['wikitext2_ppl']:.3f} | decode {res['decode_tokens_per_s']:.1f} tok/s | "
          f"weights {res['weights_gib']:.2f} GiB | peak GPU {res['peak_gpu_gib']:.2f} GiB | disk {res['disk_gib']:.2f} GiB")
'''
(ROOT / "utils" / "evalkit.py").write_text(EVALKIT_SRC)
import importlib, evalkit
importlib.reload(evalkit)
print("evalkit written:", list(evalkit.TEMPLATES))

evalkit written: ['primary', 'plain', 'question_blank', 'instruction']


In [14]:
import json, os
import pandas as pd
from pathlib import Path
from datasets import load_dataset

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
CFG = frozen["config"]; dom = CFG["domain"]; SEED = CFG["seeds"]["data"]
DEV_EVAL_N = 250

dev = pd.read_json(ROOT / "data" / f"dev_pool_{dom['name']}.jsonl", lines=True)
raw = load_dataset(dom["hf_dataset"], dom["hf_config"], split=dom["split"])

dev_eval = dev.sample(n=DEV_EVAL_N, random_state=SEED + 1).sort_values("src_idx").reset_index(drop=True)
dev_rest = dev[~dev["uid"].isin(dev_eval["uid"])].reset_index(drop=True)

# Attach the raw question and choices (needed for option shifts and alternate templates)
qs, chs = [], []
for src_idx, prompt in zip(dev_eval["src_idx"], dev_eval["question"]):
    r = raw[int(src_idx)]
    assert utils.format_mcq(r["question"], list(r["choices"]), dom["subject_string"]) == prompt, "prompt mismatch"
    qs.append(r["question"]); chs.append(list(r["choices"]))
dev_eval["q"], dev_eval["choices"] = qs, chs

def write_once(df, path):
    path = Path(path)
    if path.exists():
        print(f"[skip] {path.name} already exists")
        return
    df.to_json(path, orient="records", lines=True)
    os.chmod(path, 0o444)
    print(f"[wrote] {path.name} ({len(df)} rows)")

write_once(dev_eval, ROOT / "data" / f"dev_eval_{dom['name']}.jsonl")
write_once(dev_rest, ROOT / "data" / f"dev_rest_{dom['name']}.jsonl")

# Fluency set: fixed once, no fallback (a different fallback text would make perplexities incomparable)
fpath = ROOT / "data" / "fluency_texts.json"
if not fpath.exists():
    wt2 = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    texts = [t.strip() for t in wt2["text"] if len(t.strip()) > 200][:64]
    assert len(texts) == 64, f"expected 64 passages, got {len(texts)}"
    fpath.write_text(json.dumps(texts))
    print("[wrote] fluency_texts.json (64 passages)")
print(f"dev_eval {len(dev_eval)} | dev_rest {len(dev_rest)} (use dev_rest, not dev_eval, for importance scoring and recovery in Stages 2-5)")

[wrote] dev_eval_wmdp-bio.jsonl (250 rows)
[wrote] dev_rest_wmdp-bio.jsonl (721 rows)


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[wrote] fluency_texts.json (64 passages)
dev_eval 250 | dev_rest 721 (use dev_rest, not dev_eval, for importance scoring and recovery in Stages 2-5)


In [15]:
!ls -R /kaggle/working/specialist | head -n 40

/kaggle/working/specialist:
artifacts
config
data
logs
metrics
utils

/kaggle/working/specialist/artifacts:

/kaggle/working/specialist/config:
criteria_frozen.json

/kaggle/working/specialist/data:
data_manifest.json
dev_eval_wmdp-bio.jsonl
dev_pool_wmdp-bio.jsonl
dev_rest_wmdp-bio.jsonl
final_test_general_mmlu.jsonl
final_test_wmdp-bio.jsonl
fluency_texts.json

/kaggle/working/specialist/logs:
final_test_ledger.json

/kaggle/working/specialist/metrics:

/kaggle/working/specialist/utils:
evalkit.py
__pycache__
utils.py

/kaggle/working/specialist/utils/__pycache__:
evalkit.cpython-313.pyc
utils.cpython-313.pyc


In [16]:
import time, torch, json, importlib
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download
import evalkit
importlib.reload(evalkit)

frozen, rows, texts, letter_ids = evalkit.load_eval_inputs(ROOT)
MODEL_ID = frozen["model_id"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# On-disk size of the original checkpoint
snap = Path(snapshot_download(MODEL_ID, allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"]))
disk_gib = sum(p.stat().st_size for p in snap.glob("*.safetensors")) / 1024**3

utils.free_gpu()
utils.assert_free_gpu(9, devices=[0])
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})
model.eval()
load_s = time.perf_counter() - t0
utils.gpu_report("fp16 loaded")

res = evalkit.run_baseline(model, tokenizer, rows, texts, letter_ids)
res.update({"variant": "base_fp16", "disk_gib": disk_gib, "load_s": load_s, "timestamp": utils.now_utc()})
(ROOT / "metrics" / "stage1_base_fp16.json").write_text(json.dumps(res, indent=2))
evalkit.print_summary(res)

del model
utils.free_gpu()
utils.gpu_report("after fp16 cleanup")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[gpu 0 fp16 loaded] free 8.22 / 14.56 GiB | allocated 6.24 | reserved 6.24
[gpu 1 fp16 loaded] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00
[base_fp16] primary acc 0.596 (Wilson 95% 0.534 to 0.655) | n=250
  option shifts: {'0': 0.596, '1': 0.62, '2': 0.608, '3': 0.604}
  templates:    {'plain': 0.628, 'question_blank': 0.66, 'instruction': 0.692}
  predicted-letter histogram (primary): [61, 78, 69, 42]
  wikitext-2 ppl 15.348 | decode 20.5 tok/s | weights 6.24 GiB | peak GPU 8.74 GiB | disk 6.24 GiB
[gpu 0 after fp16 cleanup] free 14.41 / 14.56 GiB | allocated 0.01 | reserved 0.02
[gpu 1 after fp16 cleanup] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00


{0: {'free_gib': 14.41094970703125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0079345703125,
  'reserved_gib': 0.01953125},
 1: {'free_gib': 14.45977783203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0,
  'reserved_gib': 0.0}}

In [18]:
import time, shutil, torch, json, importlib
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evalkit
importlib.reload(evalkit)

frozen, rows, texts, letter_ids = evalkit.load_eval_inputs(ROOT)
MODEL_ID = frozen["model_id"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=False)
utils.free_gpu()
utils.assert_free_gpu(5, devices=[0])
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             torch_dtype=torch.float16, device_map={"": 0})
model.eval()
load_s = time.perf_counter() - t0
utils.gpu_report("4-bit loaded")

res = evalkit.run_baseline(model, tokenizer, rows, texts, letter_ids)

# Disk size of the quantized model, measured by actually saving it (the temp copy is deleted)
q_dir = ROOT / "artifacts" / "base_4bit_tmp"
try:
    model.save_pretrained(q_dir)
    disk_gib = sum(f.stat().st_size for f in q_dir.rglob("*") if f.is_file()) / 1024**3
    disk_note = "measured from save_pretrained"
except Exception as e:
    disk_gib = model.get_memory_footprint() / 1024**3
    disk_note = f"save failed ({type(e).__name__}); used in-memory footprint instead"
finally:
    shutil.rmtree(q_dir, ignore_errors=True)

res.update({"variant": "base_4bit_nf4", "disk_gib": disk_gib, "disk_note": disk_note,
            "load_s": load_s, "timestamp": utils.now_utc()})
(ROOT / "metrics" / "stage1_base_4bit.json").write_text(json.dumps(res, indent=2))
evalkit.print_summary(res)
print("disk:", disk_note)

del model
utils.free_gpu()
utils.gpu_report("after 4-bit cleanup")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[gpu 0 4-bit loaded] free 11.88 / 14.56 GiB | allocated 2.49 | reserved 2.55
[gpu 1 4-bit loaded] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[base_4bit_nf4] primary acc 0.604 (Wilson 95% 0.542 to 0.663) | n=250
  option shifts: {'0': 0.604, '1': 0.636, '2': 0.6, '3': 0.568}
  templates:    {'plain': 0.64, 'question_blank': 0.644, 'instruction': 0.644}
  predicted-letter histogram (primary): [44, 97, 67, 42]
  wikitext-2 ppl 16.045 | decode 15.5 tok/s | weights 2.31 GiB | peak GPU 4.99 GiB | disk 2.47 GiB
disk: measured from save_pretrained
[gpu 0 after 4-bit cleanup] free 14.41 / 14.56 GiB | allocated 0.01 | reserved 0.02
[gpu 1 after 4-bit cleanup] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00


{0: {'free_gib': 14.41094970703125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0079345703125,
  'reserved_gib': 0.01953125},
 1: {'free_gib': 14.45977783203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0,
  'reserved_gib': 0.0}}

In [19]:
import json
import pandas as pd

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
crit = frozen["config"]["criteria"]
a = json.loads((ROOT / "metrics" / "stage1_base_fp16.json").read_text())
b = json.loads((ROOT / "metrics" / "stage1_base_4bit.json").read_text())

def robustness_gap_pts(r):
    accs = [v["acc"] for v in r["shifts"].values()] + [v["acc"] for v in r["templates"].values()]
    return 100 * max(abs(x - r["primary_acc"]) for x in accs)

table = pd.DataFrame([{
    "variant": r["variant"], "disk_GiB": round(r["disk_gib"], 2), "weights_GiB": round(r["weights_gib"], 2),
    "peak_GPU_GiB": round(r["peak_gpu_gib"], 2), "acc": round(r["primary_acc"], 3),
    "ppl": round(r["wikitext2_ppl"], 2), "decode_tok_s": round(r["decode_tokens_per_s"], 1),
    "max_robust_gap_pts": round(robustness_gap_pts(r), 1),
} for r in (a, b)])
print(table.to_string(index=False))

diff, (lo, hi) = utils.paired_bootstrap_diff(a["primary_correct"], b["primary_correct"], seed=frozen["config"]["seeds"]["bootstrap"])
print(f"\nfp16 minus 4-bit accuracy: {diff * 100:+.1f} pts (paired 95% CI {lo * 100:+.1f} to {hi * 100:+.1f}) on n={a['n']}")
print(f"4-bit vs fp16 speed: {b['decode_tokens_per_s'] / a['decode_tokens_per_s']:.2f}x | perplexity ratio: {b['wikitext2_ppl'] / a['wikitext2_ppl']:.3f}")

print("\n=== Gate ===")
size_ok = b["disk_gib"] <= crit["max_disk_gib"]
print(f"4-bit base disk {b['disk_gib']:.2f} GiB vs target {crit['max_disk_gib']} GiB -> "
      + ("already meets the size target" if size_ok else "does NOT meet the size target, so compression is needed"))
print(f"base robustness gap {robustness_gap_pts(a):.1f} pts (fp16) / {robustness_gap_pts(b):.1f} pts (4-bit) "
      f"vs the {crit['max_robustness_gap_pts']} pt criterion: if the base itself exceeds it, this criterion needs to be relative to the base.")
print("Note: speed here is a T4 proxy (bitsandbytes needs a GPU). Real target-hardware speed comes from the Stage 7 export.")

      variant  disk_GiB  weights_GiB  peak_GPU_GiB   acc   ppl  decode_tok_s  max_robust_gap_pts
    base_fp16      6.24         6.24          8.74 0.596 15.35          20.5                 9.6
base_4bit_nf4      2.47         2.31          4.99 0.604 16.05          15.5                 4.0

fp16 minus 4-bit accuracy: -0.8 pts (paired 95% CI -4.8 to +3.2) on n=250
4-bit vs fp16 speed: 0.76x | perplexity ratio: 1.045

=== Gate ===
4-bit base disk 2.47 GiB vs target 2.0 GiB -> does NOT meet the size target, so compression is needed
base robustness gap 9.6 pts (fp16) / 4.0 pts (4-bit) vs the 5.0 pt criterion: if the base itself exceeds it, this criterion needs to be relative to the base.
Note: speed here is a T4 proxy (bitsandbytes needs a GPU). Real target-hardware speed comes from the Stage 7 export.


In [20]:
import os
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError

def get_write_token():
    tok = os.environ.get("HF_WRITE_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
        except Exception:
            tok = None
    return tok

def backup_to_hf(root=ROOT, repo_name="specialist-backup"):
    tok = get_write_token()
    if not tok:
        raise RuntimeError("No HF_WRITE_TOKEN secret found. Add it in Kaggle (Add-ons > Secrets) and attach it to this notebook.")
    api = HfApi(token=tok)
    who = api.whoami()
    auth = who.get("auth", {}).get("accessToken", {})
    print(f"[backup] user: {who['name']} | token role: {auth.get('role')} | name: {auth.get('displayName')}")
    repo_id = f"{who['name']}/{repo_name}"
    try:
        api.repo_info(repo_id, repo_type="dataset")
    except RepositoryNotFoundError:
        api.create_repo(repo_id=repo_id, repo_type="dataset", private=True)
    api.upload_folder(folder_path=str(root), repo_id=repo_id, repo_type="dataset",
                      ignore_patterns=["artifacts/*", "**/__pycache__/*"],
                      commit_message=f"backup {utils.now_utc()}")
    print(f"[backup] uploaded {root} to private dataset {repo_id}")

try:
    backup_to_hf()
except Exception as e:
    print(f"[backup] FAILED: {type(e).__name__}: {str(e)[:300]}")

[backup] user: Anirudh-Peddinti | token role: write | name: kaggle-write
[backup] uploaded /kaggle/working/specialist to private dataset Anirudh-Peddinti/specialist-backup


In [21]:
import json
for f in ["stage1_base_fp16.json", "stage1_base_4bit.json"]:
    r = json.loads((ROOT / "metrics" / f).read_text())
    print(r["variant"], "| primary", round(r["primary_acc"], 3), "| letters", r["primary_pred_hist"])
    print("  shifts:   ", {k: (round(v["acc"], 3), v["pred_hist"]) for k, v in r["shifts"].items()})
    print("  templates:", {k: (round(v["acc"], 3), v["pred_hist"]) for k, v in r["templates"].items()})

base_fp16 | primary 0.596 | letters [61, 78, 69, 42]
  shifts:    {'0': (0.596, [61, 78, 69, 42]), '1': (0.62, [63, 103, 51, 33]), '2': (0.608, [79, 82, 47, 42]), '3': (0.604, [67, 81, 53, 49])}
  templates: {'plain': (0.628, [49, 62, 92, 47]), 'question_blank': (0.66, [50, 56, 84, 60]), 'instruction': (0.692, [46, 55, 90, 59])}
base_4bit_nf4 | primary 0.604 | letters [44, 97, 67, 42]
  shifts:    {'0': (0.604, [44, 97, 67, 42]), '1': (0.636, [50, 113, 54, 33]), '2': (0.6, [67, 99, 45, 39]), '3': (0.568, [54, 98, 56, 42])}
  templates: {'plain': (0.64, [31, 70, 101, 48]), 'question_blank': (0.644, [36, 66, 84, 64]), 'instruction': (0.644, [37, 65, 94, 54])}


## Stage 2

In [22]:
import json, importlib
from pathlib import Path
import utils

try:
    ROOT
except NameError:
    raise RuntimeError("Run Cell 0.1 first: it defines ROOT and utils.")

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
name = frozen["config"]["domain"]["name"]
for need in [f"data/dev_rest_{name}.jsonl", f"data/dev_eval_{name}.jsonl", f"data/final_test_{name}.jsonl",
             "data/final_test_general_mmlu.jsonl", "data/data_manifest.json", "data/fluency_texts.json"]:
    assert (ROOT / need).exists(), f"missing {need}: restore from the HF backup (snapshot_download) or re-run Stages 0-1"

DATAKIT_SRC = r'''
import json, os, re
from pathlib import Path
import numpy as np
import pandas as pd
import utils

def read_jsonl(path):
    with open(path) as f:
        return pd.DataFrame([json.loads(line) for line in f])

def write_sealed(df, path):
    path = Path(path)
    if path.exists():
        tmp = path.with_suffix(".tmp")
        df.to_json(tmp, orient="records", lines=True)
        same = utils.sha256_file(tmp) == utils.sha256_file(path)
        tmp.unlink()
        assert same, (f"{path.name} exists but regenerating gives different content "
                      "(library versions, seed or inputs changed). Do not overwrite a sealed file.")
        print(f"[sealed] {path.name} already exists and reproduces exactly")
    else:
        df.to_json(path, orient="records", lines=True)
        os.chmod(path, 0o444)
        print(f"[sealed] wrote {path.name} ({len(df)} rows, read-only)")
    return utils.sha256_file(path)

# Drops the header line and the final "Answer:" line, keeping question and options
def strip_prompt(p):
    lines = p.split("\n")
    return " ".join(lines[2:-1])

def words(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def ngrams(ws, n):
    return {" ".join(ws[i:i + n]) for i in range(len(ws) - n + 1)}

# Pairs each question with the length-closest question whose answer letter differs
def build_pairs(df):
    lengths = df["token_len"].to_numpy()
    answers = df["answer"].to_numpy()
    out = []
    for i in range(len(df)):
        cand = np.where(answers != answers[i])[0]
        j = int(cand[np.argmin(np.abs(lengths[cand] - lengths[i]))])
        c, k = df.iloc[i], df.iloc[j]
        out.append({"uid": c["uid"], "question": c["question"], "answer": c["answer"],
                    "answer_idx": int(c["answer_idx"]), "token_len": int(c["token_len"]),
                    "corrupt_uid": k["uid"], "corrupt_question": k["question"],
                    "corrupt_answer": k["answer"], "corrupt_answer_idx": int(k["answer_idx"]),
                    "corrupt_token_len": int(k["token_len"]),
                    "abs_len_diff": int(abs(c["token_len"] - k["token_len"]))})
    return pd.DataFrame(out)
'''
(ROOT / "utils" / "datakit.py").write_text(DATAKIT_SRC)
import datakit
importlib.reload(datakit)

DESIGN_PATH = ROOT / "config" / "design_stage2.json"
FORCE_REDESIGN = False

# ================= EDIT, THEN RUN ONCE =================
DESIGN = {
    "goal": "Proof of concept: mech-interp-guided selection yields a smaller, domain-specialized model "
            "that runs locally on CPU, with prompts staying on the device.",
    "arms": ["random", "standard_importance", "circuit_guided"],
    "matched_budget": {
        "target_param_reduction_pct": 40.0,    # choose BEFORE seeing results; arms must match within 1%
        "note": "Stage 4 removes the same parameter count from every arm.",
    },
    "recovery": {"method": "teacher-KL distillation from the unmodified fp16 base",
                 "uses_gold_labels": False, "identical_for_all_arms": True},
    "claim_rule": "Mech interp is credited only if circuit_guided beats BOTH controls "
                  "(paired-bootstrap 95% CI excluding 0) on the sealed final test.",
    "primary_metrics": ["domain accuracy retained", "off-domain accuracy reduced"],
    "report_only": ["parameter count", "file size", "RAM", "cpu_tokens_per_s",
                    "wikitext2_ppl", "robustness gaps", "general_mmlu_accuracy"],
    "split_sizes": {"dom_mcq_disc": 192, "dom_mcq_val": 160, "gen_mcq_disc": 160, "gen_mcq_val": 200},
    "text": {
        "domain_source": "qiaojin/PubMedQA / pqa_unlabeled",
        "general_source": "Salesforce/wikitext / wikitext-2-raw-v1 / train",
        "domain_text_n_train": 8000, "domain_text_n_score": 512, "domain_text_n_val": 256,
        "general_text_n_score": 512, "general_text_n_val": 256,
        "min_chars": 400, "contamination_ngram": 8,
    },
    "domain_adjacent_mmlu_subjects": [
        "anatomy", "clinical_knowledge", "college_biology", "college_medicine", "high_school_biology",
        "medical_genetics", "nutrition", "professional_medicine", "virology", "human_aging"],
    "seed": frozen["config"]["seeds"]["data"] + 100,
}
# =======================================================

h = utils.sha256_json(DESIGN)
if DESIGN_PATH.exists() and not FORCE_REDESIGN:
    old = json.loads(DESIGN_PATH.read_text())
    if old["design_sha256"] != h:
        raise RuntimeError("design_stage2.json is already frozen and your DESIGN differs. Revert your edits, "
                           "or set FORCE_REDESIGN=True and accept that this is a deliberate change.")
    print(f"[design] already frozen at {old['frozen_at']} (sha {old['design_sha256'][:12]}), unchanged")
else:
    if DESIGN_PATH.exists():
        DESIGN_PATH.rename(DESIGN_PATH.with_name(f"design_stage2_archived_{utils.now_utc().replace(':', '-')}.json"))
    DESIGN_PATH.write_text(json.dumps({"frozen_at": utils.now_utc(), "design_sha256": h, "design": DESIGN}, indent=2))
    print(f"[design] wrote {DESIGN_PATH.name} (sha {h[:12]})")
print("criteria_frozen.json is untouched; its disk, speed and robustness thresholds are reference-only for this design.")
print("Power note: with n=300 on the sealed test, differences between arms under about 5 points are unlikely to be resolved.")

[design] wrote design_stage2.json (sha 7a54b154eda1)
criteria_frozen.json is untouched; its disk, speed and robustness thresholds are reference-only for this design.
Power note: with n=300 on the sealed test, differences between arms under about 5 points are unlikely to be resolved.


In [23]:
import numpy as np, pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
import datakit, utils

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
design = json.loads((ROOT / "config" / "design_stage2.json").read_text())["design"]
name, seed = frozen["config"]["domain"]["name"], design["seed"]
MAX_TOK = frozen["config"]["data"]["max_tokens"]
sz = design["split_sizes"]
D = ROOT / "data"

# ---- domain MCQs from dev_rest ----
dev_rest = datakit.read_jsonl(D / f"dev_rest_{name}.jsonl")
sh = dev_rest.sample(frac=1.0, random_state=seed).reset_index(drop=True)
dom_disc = sh.iloc[:sz["dom_mcq_disc"]].reset_index(drop=True)
dom_val = sh.iloc[sz["dom_mcq_disc"]:sz["dom_mcq_disc"] + sz["dom_mcq_val"]].reset_index(drop=True)
dom_train = sh.iloc[sz["dom_mcq_disc"] + sz["dom_mcq_val"]:].reset_index(drop=True)
assert len(dom_train) >= 200, f"dom_train has only {len(dom_train)} rows"
# Recovery prompts carry NO labels, so recovery cannot learn the benchmark's answers
dom_train_nolabels = dom_train[["uid", "question", "token_len"]]

# ---- general MCQs from MMLU test (non-adjacent subjects, not in the sealed general set) ----
tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
raw_g = load_dataset("cais/mmlu", "all", split="test")
sealed_gen = datakit.read_jsonl(D / "final_test_general_mmlu.jsonl")
sealed_uids = set(sealed_gen["uid"])
adjacent = set(design["domain_adjacent_mmlu_subjects"])

rows, seen = [], set()
for src_idx, r in enumerate(raw_g):
    if r["subject"] in adjacent:
        continue
    choices = list(r["choices"])
    if len(choices) != 4 or int(r["answer"]) not in (0, 1, 2, 3):
        continue
    subj = r["subject"].replace("_", " ")
    prompt = utils.format_mcq(r["question"], choices, subj)
    uid = f"mmlu_{utils.sha256_text(prompt)[:16]}"
    if uid in sealed_uids or uid in seen:
        continue
    seen.add(uid)
    n_tok = len(tokenizer.encode(prompt, add_special_tokens=True))
    if n_tok > MAX_TOK:
        continue
    a = int(r["answer"])
    rows.append({"uid": uid, "src_idx": src_idx, "subject": subj, "question": prompt, "answer_idx": a,
                 "answer": utils.LETTERS[a], "token_len": n_tok})
df_g = pd.DataFrame(rows)
need_g = sz["gen_mcq_disc"] + sz["gen_mcq_val"]
assert len(df_g) >= need_g, f"only {len(df_g)} eligible general rows"
gs = df_g.sample(n=need_g, random_state=seed + 1).reset_index(drop=True)
gen_disc = gs.iloc[:sz["gen_mcq_disc"]].reset_index(drop=True)
gen_val = gs.iloc[sz["gen_mcq_disc"]:].reset_index(drop=True)

# ---- counterfactual pairs for EAP-IG (Stage 3) ----
dom_pairs, gen_pairs = datakit.build_pairs(dom_disc), datakit.build_pairs(gen_disc)

# ---- write ----
outs = {
    "dom_mcq_disc": dom_disc, "dom_mcq_val": dom_val, "dom_mcq_train_nolabels": dom_train_nolabels,
    "gen_mcq_disc": gen_disc, "gen_mcq_val": gen_val, "dom_pairs": dom_pairs, "gen_pairs": gen_pairs,
}
hashes = {f"{k}.jsonl": datakit.write_sealed(v, D / f"{k}.jsonl") for k, v in outs.items()}

# ---- summary ----
for k, v in outs.items():
    extra = ""
    if "answer" in v.columns:
        extra = f" | answers {v['answer'].value_counts().sort_index().to_dict()}"
    print(f"{k:24s} n={len(v):4d} | mean tokens {v['token_len'].mean():.0f}{extra}")
print(f"pairs: mean length diff dom {dom_pairs['abs_len_diff'].mean():.1f} / gen {gen_pairs['abs_len_diff'].mean():.1f} tokens")
adj_in_sealed = sum(s in {a.replace('_', ' ') for a in adjacent} for s in sealed_gen["subject"])
print(f"sealed general set: {adj_in_sealed} of {len(sealed_gen)} rows are bio/medical-adjacent; "
      f"Stage 8 should report the other {len(sealed_gen) - adj_in_sealed} as the off-domain subset")

[sealed] wrote dom_mcq_disc.jsonl (192 rows, read-only)
[sealed] wrote dom_mcq_val.jsonl (160 rows, read-only)
[sealed] wrote dom_mcq_train_nolabels.jsonl (369 rows, read-only)
[sealed] wrote gen_mcq_disc.jsonl (160 rows, read-only)
[sealed] wrote gen_mcq_val.jsonl (200 rows, read-only)
[sealed] wrote dom_pairs.jsonl (192 rows, read-only)
[sealed] wrote gen_pairs.jsonl (160 rows, read-only)
dom_mcq_disc             n= 192 | mean tokens 102 | answers {'A': 46, 'B': 48, 'C': 64, 'D': 34}
dom_mcq_val              n= 160 | mean tokens 99 | answers {'A': 39, 'B': 42, 'C': 41, 'D': 38}
dom_mcq_train_nolabels   n= 369 | mean tokens 100
gen_mcq_disc             n= 160 | mean tokens 108 | answers {'A': 26, 'B': 47, 'C': 49, 'D': 38}
gen_mcq_val              n= 200 | mean tokens 118 | answers {'A': 53, 'B': 54, 'C': 42, 'D': 51}
dom_pairs                n= 192 | mean tokens 102 | answers {'A': 46, 'B': 48, 'C': 64, 'D': 34}
gen_pairs                n= 160 | mean tokens 108 | answers {'A': 26, 'B

In [24]:
import re
import numpy as np, pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer
import datakit, utils

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
design = json.loads((ROOT / "config" / "design_stage2.json").read_text())["design"]
name, seed, tc = frozen["config"]["domain"]["name"], design["seed"], design["text"]
D = ROOT / "data"

# ---- candidate passages ----
pq = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")
passages, seen = [], set()
for r in pq:
    ctx = r["context"]["contexts"] if isinstance(r["context"], dict) else r["context"]
    text = " ".join(" ".join(ctx).split() + (r.get("long_answer") or "").split())
    if len(text) < tc["min_chars"]:
        continue
    h = utils.sha256_text(text)
    if h in seen:
        continue
    seen.add(h)
    passages.append({"pubid": int(r["pubid"]), "text": text})
print(f"{len(passages)} candidate abstracts")

# ---- rank by similarity to the DEV domain questions only (never dev_eval or the sealed test) ----
q_df = pd.concat([datakit.read_jsonl(D / "dom_mcq_disc.jsonl")["question"],
                  datakit.read_jsonl(D / "dom_mcq_train_nolabels.jsonl")["question"]])
query_texts = [datakit.strip_prompt(p) for p in q_df]
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=3, max_df=0.5,
                      max_features=60000, dtype=np.float32)
X = vec.fit_transform([p["text"] for p in passages])
Q = vec.transform(query_texts)
centroid = np.asarray(Q.mean(axis=0)).ravel()
centroid /= (np.linalg.norm(centroid) + 1e-9)
sims = np.asarray(X @ centroid).ravel()
terms = np.array(vec.get_feature_names_out())
print("top centroid terms (sanity check the domain match):", terms[np.argsort(-centroid)[:25]].tolist())

# ---- contamination filter: drop passages sharing an n-gram with ANY evaluation question ----
N = tc["contamination_ngram"]
eval_prompts = (list(datakit.read_jsonl(D / f"final_test_{name}.jsonl")["question"])
                + list(datakit.read_jsonl(D / f"dev_eval_{name}.jsonl")["question"])
                + list(datakit.read_jsonl(D / "dom_mcq_val.jsonl")["question"]))
bad = set()
for p in eval_prompts:
    bad |= datakit.ngrams(datakit.words(datakit.strip_prompt(p)), N)

need = tc["domain_text_n_train"] + tc["domain_text_n_score"] + tc["domain_text_n_val"]
kept, dropped = [], 0
for idx in np.argsort(-sims):
    if datakit.ngrams(datakit.words(passages[idx]["text"]), N) & bad:
        dropped += 1
        continue
    kept.append(int(idx))
    if len(kept) == need:
        break
assert len(kept) == need, f"only {len(kept)} usable passages"
print(f"selected {need} passages in rank order; dropped {dropped} for {N}-gram overlap with evaluation questions")

# ---- random split (so the train split is not strictly lower-similarity than the scoring split) ----
rng = np.random.default_rng(seed + 2)
rng.shuffle(kept)
n_s, n_v = tc["domain_text_n_score"], tc["domain_text_n_val"]
def frame(ids):
    return pd.DataFrame([{"pubid": passages[i]["pubid"], "text": passages[i]["text"], "sim": round(float(sims[i]), 6)} for i in ids])
parts = {"dom_text_score": frame(kept[:n_s]), "dom_text_val": frame(kept[n_s:n_s + n_v]), "dom_text_train": frame(kept[n_s + n_v:])}
hashes = {f"{k}.jsonl": datakit.write_sealed(v, D / f"{k}.jsonl") for k, v in parts.items()}

# ---- stats ----
tok = AutoTokenizer.from_pretrained(frozen["model_id"])
sample = parts["dom_text_train"]["text"].iloc[:300].tolist()
mean_tok = float(np.mean([len(tok.encode(t, add_special_tokens=False)) for t in sample]))
print(f"similarity: selected median {np.median(sims[kept]):.3f} vs corpus median {np.median(sims):.3f}")
print(f"mean tokens per passage about {mean_tok:.0f} -> train set about {mean_tok * len(parts['dom_text_train']) / 1e6:.2f}M tokens (before truncation)")
for k, v in parts.items():
    print(f"{k:16s} n={len(v)}")

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_unlabeled/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 66.0MB            

pqa_unlabeled/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/61249 [00:00<?, ? examples/s]

61233 candidate abstracts
top centroid terms (sanity check the domain match): ['viral', 'virus', 'host', 'genome', 'replication', 'cells', 'gene', 'viruses', 'cell', 'protein', 'rna', 'vectors', 'vector', 'immune', 'proteins', 'genetics', 'genes', 'reverse', 'dna', 'expression', 'following', 'capsid', 'influenza', 'mutations', 'human']
selected 8768 passages in rank order; dropped 1 for 8-gram overlap with evaluation questions
[sealed] wrote dom_text_score.jsonl (512 rows, read-only)
[sealed] wrote dom_text_val.jsonl (256 rows, read-only)
[sealed] wrote dom_text_train.jsonl (8000 rows, read-only)
similarity: selected median 0.041 vs corpus median 0.019
mean tokens per passage about 363 -> train set about 2.90M tokens (before truncation)
dom_text_score   n=512
dom_text_val     n=256
dom_text_train   n=8000


In [25]:
import itertools
import numpy as np, pandas as pd
from datasets import load_dataset
import datakit, utils

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
design = json.loads((ROOT / "config" / "design_stage2.json").read_text())["design"]
name, seed, tc = frozen["config"]["domain"]["name"], design["seed"], design["text"]
D = ROOT / "data"

# ---- general text from WikiText-2 TRAIN (Stage 1's fluency set is from the test split) ----
wt = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
texts, seen = [], set()
for t in wt["text"]:
    t = " ".join(t.split())
    if len(t) > 200 and utils.sha256_text(t) not in seen:
        seen.add(utils.sha256_text(t))
        texts.append(t)
need_g = tc["general_text_n_score"] + tc["general_text_n_val"]
assert len(texts) >= need_g
rng = np.random.default_rng(seed + 3)
pick = rng.permutation(len(texts))[:need_g]
gen_score = pd.DataFrame({"text": [texts[i] for i in pick[:tc["general_text_n_score"]]]})
gen_val = pd.DataFrame({"text": [texts[i] for i in pick[tc["general_text_n_score"]:]]})
fluency = set(json.loads((D / "fluency_texts.json").read_text()))
assert not (set(gen_score["text"]) | set(gen_val["text"])) & {" ".join(f.split()) for f in fluency}, \
    "general text overlaps the Stage 1 fluency set"
hashes = {"gen_text_score.jsonl": datakit.write_sealed(gen_score, D / "gen_text_score.jsonl"),
          "gen_text_val.jsonl": datakit.write_sealed(gen_val, D / "gen_text_val.jsonl")}

# ---- leakage checks across every MCQ split ----
uid_sets = {
    "dom_disc": set(datakit.read_jsonl(D / "dom_mcq_disc.jsonl")["uid"]),
    "dom_val": set(datakit.read_jsonl(D / "dom_mcq_val.jsonl")["uid"]),
    "dom_train": set(datakit.read_jsonl(D / "dom_mcq_train_nolabels.jsonl")["uid"]),
    "dev_eval": set(datakit.read_jsonl(D / f"dev_eval_{name}.jsonl")["uid"]),
    "final_test": set(datakit.read_jsonl(D / f"final_test_{name}.jsonl")["uid"]),
    "gen_disc": set(datakit.read_jsonl(D / "gen_mcq_disc.jsonl")["uid"]),
    "gen_val": set(datakit.read_jsonl(D / "gen_mcq_val.jsonl")["uid"]),
    "final_general": set(datakit.read_jsonl(D / "final_test_general_mmlu.jsonl")["uid"]),
}
for (a, sa), (b, sb) in itertools.combinations(uid_sets.items(), 2):
    assert not (sa & sb), f"LEAK: {a} and {b} share {len(sa & sb)} uids"
print("[check] all MCQ splits are pairwise disjoint")

txt_sets = {k: set(datakit.read_jsonl(D / f"{k}.jsonl")["text"]) for k in
            ["dom_text_score", "dom_text_val", "dom_text_train", "gen_text_score", "gen_text_val"]}
for (a, sa), (b, sb) in itertools.combinations(txt_sets.items(), 2):
    assert not (sa & sb), f"LEAK: {a} and {b} share passages"
print("[check] all text splits are pairwise disjoint")

# ---- manifest ----
files = ["dom_mcq_disc", "dom_mcq_val", "dom_mcq_train_nolabels", "gen_mcq_disc", "gen_mcq_val", "dom_pairs", "gen_pairs",
         "dom_text_score", "dom_text_val", "dom_text_train", "gen_text_score", "gen_text_val"]
manifest = {"created_at": utils.now_utc(), "design_sha256": json.loads((ROOT / "config" / "design_stage2.json").read_text())["design_sha256"],
            "files": {f"{f}.jsonl": {"rows": int(len(datakit.read_jsonl(D / f'{f}.jsonl'))),
                                      "sha256": utils.sha256_file(D / f"{f}.jsonl")} for f in files}}
(D / "stage2_manifest.json").write_text(json.dumps(manifest, indent=2))
print(pd.DataFrame([{"file": k, "rows": v["rows"]} for k, v in manifest["files"].items()]).to_string(index=False))
print("Stage 2 complete. Run the backup cell now.")

[sealed] wrote gen_text_score.jsonl (512 rows, read-only)
[sealed] wrote gen_text_val.jsonl (256 rows, read-only)
[check] all MCQ splits are pairwise disjoint
[check] all text splits are pairwise disjoint
                        file  rows
          dom_mcq_disc.jsonl   192
           dom_mcq_val.jsonl   160
dom_mcq_train_nolabels.jsonl   369
          gen_mcq_disc.jsonl   160
           gen_mcq_val.jsonl   200
             dom_pairs.jsonl   192
             gen_pairs.jsonl   160
        dom_text_score.jsonl   512
          dom_text_val.jsonl   256
        dom_text_train.jsonl  8000
        gen_text_score.jsonl   512
          gen_text_val.jsonl   256
Stage 2 complete. Run the backup cell now.


In [26]:
import os
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError

def get_write_token():
    tok = os.environ.get("HF_WRITE_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
        except Exception:
            tok = None
    return tok

def backup_to_hf(root=ROOT, repo_name="specialist-backup"):
    tok = get_write_token()
    if not tok:
        raise RuntimeError("No HF_WRITE_TOKEN secret found. Add it in Kaggle (Add-ons > Secrets) and attach it to this notebook.")
    api = HfApi(token=tok)
    who = api.whoami()
    auth = who.get("auth", {}).get("accessToken", {})
    print(f"[backup] user: {who['name']} | token role: {auth.get('role')} | name: {auth.get('displayName')}")
    repo_id = f"{who['name']}/{repo_name}"
    try:
        api.repo_info(repo_id, repo_type="dataset")
    except RepositoryNotFoundError:
        api.create_repo(repo_id=repo_id, repo_type="dataset", private=True)
    api.upload_folder(folder_path=str(root), repo_id=repo_id, repo_type="dataset",
                      ignore_patterns=["artifacts/*", "**/__pycache__/*"],
                      commit_message=f"backup {utils.now_utc()}")
    print(f"[backup] uploaded {root} to private dataset {repo_id}")

try:
    backup_to_hf()
except Exception as e:
    print(f"[backup] FAILED: {type(e).__name__}: {str(e)[:300]}")

[backup] user: Anirudh-Peddinti | token role: write | name: kaggle-write
[backup] uploaded /kaggle/working/specialist to private dataset Anirudh-Peddinti/specialist-backup


In [27]:
for n in ["dom_pairs", "gen_pairs"]:
    d = datakit.read_jsonl(ROOT / "data" / f"{n}.jsonl")
    print(n, "clean:", d["answer"].value_counts().sort_index().to_dict(),
          "| corrupt:", d["corrupt_answer"].value_counts().sort_index().to_dict())

dom_pairs clean: {'A': 46, 'B': 48, 'C': 64, 'D': 34} | corrupt: {'A': 56, 'B': 48, 'C': 48, 'D': 40}
gen_pairs clean: {'A': 26, 'B': 47, 'C': 49, 'D': 38} | corrupt: {'A': 33, 'B': 40, 'C': 56, 'D': 31}


In [28]:
import json
import numpy as np, pandas as pd
from scipy.optimize import linear_sum_assignment
import datakit, utils

D = ROOT / "data"

def build_pairs_balanced(df):
    ans, ln = df["answer"].to_numpy(), df["token_len"].to_numpy()
    cost = np.abs(ln[:, None] - ln[None, :]).astype(float)
    cost[ans[:, None] == ans[None, :]] = 1e6      # the corrupt partner must have a different answer letter
    rows, cols = linear_sum_assignment(cost)      # a permutation: each question is a corrupt partner exactly once
    assert cost[rows, cols].max() < 1e6, "no valid balanced pairing"
    out = []
    for i, j in zip(rows, cols):
        c, k = df.iloc[i], df.iloc[j]
        out.append({"uid": c["uid"], "question": c["question"], "answer": c["answer"],
                    "answer_idx": int(c["answer_idx"]), "token_len": int(c["token_len"]),
                    "corrupt_uid": k["uid"], "corrupt_question": k["question"],
                    "corrupt_answer": k["answer"], "corrupt_answer_idx": int(k["answer_idx"]),
                    "corrupt_token_len": int(k["token_len"]),
                    "abs_len_diff": int(abs(c["token_len"] - k["token_len"]))})
    return pd.DataFrame(out)

hashes = {}
for src, dst in [("dom_mcq_disc", "dom_pairs_balanced"), ("gen_mcq_disc", "gen_pairs_balanced")]:
    pairs = build_pairs_balanced(datakit.read_jsonl(D / f"{src}.jsonl"))
    print(dst, "clean:", pairs["answer"].value_counts().sort_index().to_dict(),
          "| corrupt:", pairs["corrupt_answer"].value_counts().sort_index().to_dict(),
          "| mean length diff", round(pairs["abs_len_diff"].mean(), 2))
    hashes[f"{dst}.jsonl"] = datakit.write_sealed(pairs, D / f"{dst}.jsonl")
(D / "stage2_pairs_addendum.json").write_text(json.dumps({"created_at": utils.now_utc(), "files": hashes}, indent=2))

dom_pairs_balanced clean: {'A': 46, 'B': 48, 'C': 64, 'D': 34} | corrupt: {'A': 46, 'B': 48, 'C': 64, 'D': 34} | mean length diff 1.3
[sealed] wrote dom_pairs_balanced.jsonl (192 rows, read-only)
gen_pairs_balanced clean: {'A': 26, 'B': 47, 'C': 49, 'D': 38} | corrupt: {'A': 26, 'B': 47, 'C': 49, 'D': 38} | mean length diff 2.89
[sealed] wrote gen_pairs_balanced.jsonl (160 rows, read-only)


271

## Stage 3

In [29]:
import json
import utils

try:
    ROOT
except NameError:
    raise RuntimeError("Run Cell 0.1 first: it defines ROOT and utils.")

design_rec = json.loads((ROOT / "config" / "design_stage2.json").read_text())
for need in ["data/dom_pairs_balanced.jsonl", "data/gen_pairs_balanced.jsonl", "data/dom_text_score.jsonl",
             "data/dom_mcq_disc.jsonl", "data/gen_mcq_disc.jsonl", "metrics/stage1_base_fp16.json"]:
    assert (ROOT / need).exists(), f"missing {need}: restore from the HF backup or re-run the earlier stage"
(ROOT / "scores").mkdir(exist_ok=True)

AMEND_PATH = ROOT / "config" / "design_stage3_amendment.json"
FORCE_AMEND = False

AMENDMENT = {
    "parent_design_sha256": design_rec["design_sha256"],
    "supersedes": "design_stage2.json matched_budget.target_param_reduction_pct (40% total) is reference-only",
    "scope": {
        "pruned": "MLP neurons (gate row, up row, down column), the same number removed in every layer",
        "not_pruned": ["attention heads", "vocabulary (trim is applied to the winning arm only, after the comparison)"],
    },
    "arms": {
        "random": "uniform random neuron scores per layer; sweep seeds 0-2, recovery seed 0",
        "standard_importance": "mean |activation| x L2 norm of the neuron's down_proj column, on the 192 dom_mcq_disc prompts",
        "circuit_guided": "zero-ablation attribution patching: mean over 192 balanced pairs of |sum_t a * dm/da|, "
                          "m = logit(correct letter) - logit(counterfactual's letter) at the answer position",
    },
    "report_only_variants": {"standard_text": "same as standard_importance on 512 dom_text_score passages"},
    "pairs_files": ["dom_pairs_balanced.jsonl", "gen_pairs_balanced.jsonl"],
    "budget_selection": {
        "grid_mlp_neuron_reduction": [0.10, 0.25, 0.40, 0.55],
        "data": "dev_eval only, never the sealed test",
        "recovery": "none (pre-recovery pruning)",
        "random_arm_seeds": [0, 1, 2],
        "min_relative_accuracy": 0.80,
        "rule": "the largest grid reduction at which the MEDIAN of the three arms' dev_eval accuracy "
                "(random arm = mean over its seeds) is >= 0.80 x the unpruned dev_eval accuracy; "
                "if none qualifies, the smallest grid value",
    },
}

h = utils.sha256_json(AMENDMENT)
if AMEND_PATH.exists() and not FORCE_AMEND:
    old = json.loads(AMEND_PATH.read_text())
    if old["amendment_sha256"] != h:
        raise RuntimeError("The amendment is already frozen and differs from your edits. Revert, or set FORCE_AMEND=True deliberately.")
    print(f"[amendment] already frozen at {old['frozen_at']}, unchanged")
else:
    if AMEND_PATH.exists():
        AMEND_PATH.rename(AMEND_PATH.with_name(f"design_stage3_amendment_archived_{utils.now_utc().replace(':', '-')}.json"))
    AMEND_PATH.write_text(json.dumps({"frozen_at": utils.now_utc(), "amendment_sha256": h, "amendment": AMENDMENT}, indent=2))
    print(f"[amendment] wrote {AMEND_PATH.name} (sha {h[:12]}) BEFORE any pruned model has been scored")

[amendment] wrote design_stage3_amendment.json (sha 3eace5591dfb) BEFORE any pruned model has been scored


In [30]:
SCOREKIT_SRC = r'''
import numpy as np
import torch
import evalkit, utils

def load_model(model_id, device=0):
    from transformers import AutoModelForCausalLM
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": device})
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m

# Zeroing the input of down_proj for neuron n is identical to zeroing gate row n, up row n and down column n.
class NeuronMasks:
    def __init__(self, model):
        self.n_layers = model.config.num_hidden_layers
        self.d_mlp = model.config.intermediate_size
        self.dev = model.get_input_embeddings().weight.device
        self.masks = [torch.ones(self.d_mlp, dtype=torch.float16, device=self.dev) for _ in range(self.n_layers)]
        self.handles = [layer.mlp.down_proj.register_forward_pre_hook(self._make(l))
                        for l, layer in enumerate(model.model.layers)]

    def _make(self, l):
        def pre(module, args):
            return (args[0] * self.masks[l],)
        return pre

    def set_keep(self, keep):
        for l in range(self.n_layers):
            m = torch.zeros(self.d_mlp, dtype=torch.float16, device=self.dev)
            m[keep[l].to(self.dev)] = 1.0
            self.masks[l] = m

    def reset(self):
        self.masks = [torch.ones(self.d_mlp, dtype=torch.float16, device=self.dev) for _ in range(self.n_layers)]

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []

class Capture:
    def __init__(self, model):
        self.acts, self.need_grad, self.handles = {}, False, []
        self.handles.append(model.get_input_embeddings().register_forward_hook(self._emb))
        for l, layer in enumerate(model.model.layers):
            self.handles.append(layer.mlp.down_proj.register_forward_pre_hook(self._make(l)))

    def _emb(self, module, args, out):
        if self.need_grad and not out.requires_grad:
            out.requires_grad_(True)       # parameters are frozen, so this is what lets gradients reach the activations
        return out

    def _make(self, l):
        def pre(module, args):
            a = args[0]
            if self.need_grad:
                if not a.requires_grad:
                    raise RuntimeError("MLP activation does not require grad: the embedding hook did not take effect")
                a.retain_grad()
            self.acts[l] = a
        return pre

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []

# contrast=None: activation statistics only. Otherwise contrast is a list of (correct_idx, counterfactual_idx) letter
# indices (0-3) and the pass also returns first-order zero-ablation attribution a * dm/da.
def run_pass(model, tokenizer, texts, letter_ids, contrast=None, batch_size=4, max_len=256, scale=8.0):
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    L, Dm = model.config.num_hidden_layers, model.config.intermediate_size
    dev = model.get_input_embeddings().weight.device
    cap = Capture(model)
    cap.need_grad = contrast is not None
    ids = torch.tensor(letter_ids, device=dev)
    S_abs = torch.zeros(L, Dm, device=dev)
    A_abs = torch.zeros(L, Dm, device=dev)
    A_sgn = torch.zeros(L, Dm, device=dev)
    n_tok, n_ex = 0, 0
    order = np.argsort([len(t) for t in texts])
    try:
        for s in range(0, len(texts), batch_size):
            idx = order[s:s + batch_size]
            enc = tokenizer([texts[i] for i in idx], return_tensors="pt", padding=True,
                            truncation=True, max_length=max_len).to(dev)
            v = enc["attention_mask"].to(torch.float32)[..., None]          # [B, T, 1]
            cap.acts.clear()
            if contrast is None:
                with torch.no_grad():
                    model.model(**enc, use_cache=False)
            else:
                with torch.enable_grad():
                    try:
                        logits = model(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :]
                    except TypeError:
                        logits = model(**enc, use_cache=False).logits[:, -1, :]
                    rows_i = torch.arange(len(idx), device=dev)
                    c = ids[torch.tensor([contrast[i][0] for i in idx], device=dev)]
                    k = ids[torch.tensor([contrast[i][1] for i in idx], device=dev)]
                    m = logits[rows_i, c].float() - logits[rows_i, k].float()
                    (m.sum() * scale).backward()     # scale guards against fp16 gradient underflow
            n_tok += int(v.sum().item())
            n_ex += len(idx)
            for l in range(L):
                a = cap.acts[l]
                af = a.detach().float()
                S_abs[l] += (af.abs() * v).sum(dim=(0, 1))
                if contrast is not None:
                    contrib = (af * a.grad.float() * v).sum(dim=1)           # [B, Dm]
                    A_abs[l] += contrib.abs().sum(dim=0)
                    A_sgn[l] += contrib.sum(dim=0)
            cap.acts.clear()
    finally:
        cap.remove()
    out = {"act_abs_mean": (S_abs / max(n_tok, 1)).cpu(), "n_tok": n_tok, "n_ex": n_ex}
    if contrast is not None:
        out["attr_abs_mean"] = (A_abs / max(n_ex, 1) / scale).cpu()
        out["attr_signed_mean"] = (A_sgn / max(n_ex, 1) / scale).cpu()
    return out

def down_col_norms(model):
    return torch.stack([l.mlp.down_proj.weight.detach().float().norm(dim=0).cpu() for l in model.model.layers])

def random_scores(L, Dm, seed):
    return torch.stack([torch.from_numpy(np.random.default_rng(seed * 1000 + l).random(Dm)).float() for l in range(L)])

# Keep the top-k neurons by score in EVERY layer (uniform width)
def keep_indices(scores, keep_frac):
    L, Dm = scores.shape
    k = int(round(Dm * keep_frac))
    return [torch.topk(scores[l], k).indices.sort().values for l in range(L)]

def eval_keep(model, masks, tokenizer, rows, texts, letter_ids, keep=None):
    if keep is None:
        masks.reset()
    else:
        masks.set_keep(keep)
    prompts, labels = evalkit.build_prompts(rows, "primary", 0)
    lg = evalkit.score_mcq(model, tokenizer, prompts, letter_ids, 8)
    acc, corr, hist = evalkit.accuracy_from_logits(lg, labels)
    return {"acc": acc, "correct": corr.tolist(), "pred_hist": hist, "ppl": evalkit.corpus_ppl(model, tokenizer, texts)}
'''
(ROOT / "utils" / "scorekit.py").write_text(SCOREKIT_SRC)
import importlib, scorekit
importlib.reload(scorekit)
print("scorekit written")

scorekit written


In [31]:
import json, time, importlib
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, utils
importlib.reload(scorekit)

SMOKE = False    # True: the 16 LONGEST prompts per pass (worst-case memory), nothing saved. Then set False for the real run.

frozen, rows, texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dom_pairs = datakit.read_jsonl(D / "dom_pairs_balanced.jsonl")
gen_pairs = datakit.read_jsonl(D / "gen_pairs_balanced.jsonl")
dom_text = datakit.read_jsonl(D / "dom_text_score.jsonl")["text"].tolist()
dom_c = [(int(a), int(b)) for a, b in zip(dom_pairs["answer_idx"], dom_pairs["corrupt_answer_idx"])]
gen_c = [(int(a), int(b)) for a, b in zip(gen_pairs["answer_idx"], gen_pairs["corrupt_answer_idx"])]

def pick(prompts, contrast=None):
    if not SMOKE:
        return prompts, contrast
    order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))[:16]
    return [prompts[i] for i in order], ([contrast[i] for i in order] if contrast else None)

utils.free_gpu()
utils.assert_free_gpu(9, devices=[0])
model = scorekit.load_model(frozen["model_id"], 0)
torch.cuda.reset_peak_memory_stats(0)

t = time.perf_counter()
p_dom, c_dom = pick(dom_pairs["question"].tolist(), dom_c)
r_dom = scorekit.run_pass(model, tokenizer, p_dom, letter_ids, contrast=c_dom, batch_size=2, max_len=512)
print(f"domain attribution + activations: {r_dom['n_ex']} prompts in {time.perf_counter() - t:.0f}s")

t = time.perf_counter()
p_gen, c_gen_sel = pick(gen_pairs["question"].tolist(), gen_c)
r_gen = scorekit.run_pass(model, tokenizer, p_gen, letter_ids, contrast=c_gen_sel, batch_size=2, max_len=512)
print(f"general attribution: {r_gen['n_ex']} prompts in {time.perf_counter() - t:.0f}s")

t = time.perf_counter()
p_txt, _ = pick(dom_text)
r_txt = scorekit.run_pass(model, tokenizer, p_txt, letter_ids, contrast=None, batch_size=4, max_len=256)
print(f"domain text activations: {r_txt['n_ex']} passages ({r_txt['n_tok']:,} tokens) in {time.perf_counter() - t:.0f}s")

col = scorekit.down_col_norms(model)
peak = torch.cuda.max_memory_allocated(0) / 1024**3
print(f"peak GPU memory {peak:.2f} GiB"
      + ("   <-- above 13 GiB: use batch_size=1 for the two attribution passes" if peak > 13.0 else ""))

scores = {
    "std_mcq": r_dom["act_abs_mean"] * col,
    "std_text": r_txt["act_abs_mean"] * col,
    "attr_dom": r_dom["attr_abs_mean"],
    "attr_dom_signed": r_dom["attr_signed_mean"],
    "attr_gen": r_gen["attr_abs_mean"],
    "col_norm": col,
    "meta": {"smoke": bool(SMOKE), "n_dom": r_dom["n_ex"], "n_gen": r_gen["n_ex"],
             "n_text_tokens": r_txt["n_tok"], "created_at": utils.now_utc()},
}
for k, v in scores.items():
    if k != "meta":
        assert torch.isfinite(v).all(), f"{k} contains NaN or inf"
assert scores["attr_dom"].sum() > 0, "attribution is all zero: gradients are not reaching the activations"
print("attr_dom mean per layer (every 5th):",
      {l: round(float(scores["attr_dom"][l].mean()), 4) for l in range(0, scores["attr_dom"].shape[0], 5)})
print(f"neurons with exactly zero attribution: {(scores['attr_dom'] == 0).float().mean().item():.4%}")
print(f"attr_dom shape {tuple(scores['attr_dom'].shape)} | std_mcq shape {tuple(scores['std_mcq'].shape)}")

if SMOKE:
    print("SMOKE run: nothing saved. If the peak memory is under about 13 GiB, set SMOKE = False and re-run.")
else:
    out_path = ROOT / "scores" / "neuron_scores.pt"
    torch.save(scores, out_path)
    print(f"saved {out_path.name} ({out_path.stat().st_size / 1e6:.1f} MB)")

del model
utils.free_gpu()
utils.gpu_report("after scoring")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

domain attribution + activations: 192 prompts in 21s
general attribution: 160 prompts in 18s
domain text activations: 512 passages (128,399 tokens) in 50s
peak GPU memory 8.93 GiB
attr_dom mean per layer (every 5th): {0: 0.0017, 5: 0.0032, 10: 0.0033, 15: 0.0029, 20: 0.0018, 25: 0.0009, 30: 0.0007, 35: 0.0013}
neurons with exactly zero attribution: 0.0000%
attr_dom shape (36, 11008) | std_mcq shape (36, 11008)
saved neuron_scores.pt (9.5 MB)
[gpu 0 after scoring] free 14.38 / 14.56 GiB | allocated 0.02 | reserved 0.04
[gpu 1 after scoring] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00


{0: {'free_gib': 14.38360595703125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.015869140625,
  'reserved_gib': 0.0390625},
 1: {'free_gib': 14.45977783203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0,
  'reserved_gib': 0.0}}

In [32]:
import json, time, importlib
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, utils
importlib.reload(scorekit)

frozen, rows, texts, letter_ids = evalkit.load_eval_inputs(ROOT)
bs = json.loads((ROOT / "config" / "design_stage3_amendment.json").read_text())["amendment"]["budget_selection"]
SCORES = ROOT / "scores" / "neuron_scores.pt"
assert SCORES.exists(), "neuron_scores.pt not found: run Cell 3.3 with SMOKE=False first (SMOKE=True saves nothing)"
S = torch.load(SCORES)
assert not S["meta"]["smoke"], "scores came from a smoke run: re-run Cell 3.3 with SMOKE=False"
L, Dm = S["attr_dom"].shape

tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

RES = ROOT / "metrics" / "stage3_sweep.json"
res = json.loads(RES.read_text()) if RES.exists() else {}

utils.free_gpu()
utils.assert_free_gpu(9, devices=[0])
model = scorekit.load_model(frozen["model_id"], 0)
masks = scorekit.NeuronMasks(model)

def run(key, keep):
    if key in res:
        return
    t = time.perf_counter()
    res[key] = scorekit.eval_keep(model, masks, tokenizer, rows, texts, letter_ids, keep)
    RES.write_text(json.dumps(res))                 # saved after every evaluation, so a crash loses nothing
    print(f"{key:36s} acc {res[key]['acc']:.3f} | ppl {res[key]['ppl']:8.2f} | {time.perf_counter() - t:.0f}s")

run("base", None)
ref = json.loads((ROOT / "metrics" / "stage1_base_fp16.json").read_text())["primary_acc"]
print(f"unpruned here {res['base']['acc']:.3f} vs Stage 1 fp16 {ref:.3f}"
      + ("" if abs(res["base"]["acc"] - ref) < 0.005 else "   <-- MISMATCH: check the hooks or the dev_eval file"))

arm_scores = {"standard_importance": S["std_mcq"], "circuit_guided": S["attr_dom"], "standard_text": S["std_text"]}
for r in bs["grid_mlp_neuron_reduction"]:
    keep_frac = 1.0 - r
    for seed in bs["random_arm_seeds"]:
        run(f"random|{r}|{seed}", scorekit.keep_indices(scorekit.random_scores(L, Dm, seed), keep_frac))
    for arm, sc in arm_scores.items():
        run(f"{arm}|{r}", scorekit.keep_indices(sc, keep_frac))

masks.remove()
del model
utils.free_gpu()
utils.gpu_report("after sweep")
print("sweep complete")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base                                 acc 0.596 | ppl    15.35 | 16s
unpruned here 0.596 vs Stage 1 fp16 0.596
random|0.1|0                         acc 0.572 | ppl    22.50 | 16s
random|0.1|1                         acc 0.540 | ppl    21.03 | 16s
random|0.1|2                         acc 0.612 | ppl    20.47 | 16s
standard_importance|0.1              acc 0.612 | ppl    18.65 | 16s
circuit_guided|0.1                   acc 0.596 | ppl    19.13 | 16s
standard_text|0.1                    acc 0.612 | ppl    18.72 | 16s
random|0.25|0                        acc 0.468 | ppl    43.71 | 16s
random|0.25|1                        acc 0.316 | ppl    40.08 | 16s
random|0.25|2                        acc 0.392 | ppl    46.71 | 16s
standard_importance|0.25             acc 0.588 | ppl    27.64 | 16s
circuit_guided|0.25                  acc 0.572 | ppl    34.35 | 16s
standard_text|0.25                   acc 0.540 | ppl    29.19 | 16s
random|0.4|0                         acc 0.224 | ppl  8589.32 | 16s
random

In [33]:
import json
import numpy as np, pandas as pd, torch
from scipy.stats import spearmanr
from transformers import AutoConfig
import evalkit, scorekit, utils

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
bs = json.loads((ROOT / "config" / "design_stage3_amendment.json").read_text())["amendment"]["budget_selection"]
res = json.loads((ROOT / "metrics" / "stage3_sweep.json").read_text())
S = torch.load(ROOT / "scores" / "neuron_scores.pt")
grid, seeds, rel = bs["grid_mlp_neuron_reduction"], bs["random_arm_seeds"], bs["min_relative_accuracy"]
base_acc = res["base"]["acc"]

table = []
for r in grid:
    rnd = [res[f"random|{r}|{s}"] for s in seeds]
    row = {"reduction": r,
           "random_mean": np.mean([x["acc"] for x in rnd]),
           "random_min": min(x["acc"] for x in rnd), "random_max": max(x["acc"] for x in rnd),
           "standard": res[f"standard_importance|{r}"]["acc"],
           "circuit": res[f"circuit_guided|{r}"]["acc"],
           "std_text(ref)": res[f"standard_text|{r}"]["acc"],
           "ppl_random": np.mean([x["ppl"] for x in rnd]),
           "ppl_standard": res[f"standard_importance|{r}"]["ppl"],
           "ppl_circuit": res[f"circuit_guided|{r}"]["ppl"]}
    row["median_of_arms"] = float(np.median([row["random_mean"], row["standard"], row["circuit"]]))
    row["passes_rule"] = row["median_of_arms"] >= rel * base_acc
    table.append(row)
df = pd.DataFrame(table)
print(f"unpruned dev_eval accuracy {base_acc:.3f}; rule threshold {rel * base_acc:.3f} (median of the three arms)")
print(df.round(3).to_string(index=False))

passing = [row["reduction"] for row in table if row["passes_rule"]]
chosen = max(passing) if passing else min(grid)

cfg = AutoConfig.from_pretrained(frozen["model_id"])
total_params = 3_349_227_520
removed = chosen * cfg.intermediate_size * 3 * cfg.hidden_size * cfg.num_hidden_layers
CHOICE = ROOT / "config" / "stage3_budget_choice.json"
if CHOICE.exists():
    print(f"\n[budget] already recorded: {json.loads(CHOICE.read_text())['chosen_mlp_neuron_reduction']} (not overwritten)")
else:
    CHOICE.write_text(json.dumps({"chosen_at": utils.now_utc(), "rule": bs["rule"],
                                  "chosen_mlp_neuron_reduction": chosen,
                                  "removed_params": int(removed), "removed_pct_of_total": round(100 * removed / total_params, 2),
                                  "passing_grid_values": passing, "table": table}, indent=2))
    print(f"\n[budget] chosen MLP neuron reduction {chosen:.0%} -> {removed / 1e6:.0f}M params removed "
          f"({100 * removed / total_params:.1f}% of the model). Recorded in {CHOICE.name}.")

# ---- sanity checks on the scores at the chosen budget ----
keep_frac = 1.0 - chosen
def kept_masks(sc):
    out = []
    for idx in scorekit.keep_indices(sc, keep_frac):
        m = torch.zeros(sc.shape[1], dtype=torch.bool)
        m[idx] = True
        out.append(m)
    return out
def mean_jaccard(a, b):
    return float(np.mean([(x & y).sum().item() / (x | y).sum().item() for x, y in zip(a, b)]))
def mean_spearman(a, b):
    return float(np.mean([spearmanr(a[l].numpy(), b[l].numpy()).correlation for l in range(a.shape[0])]))

kc, ks, kt = kept_masks(S["attr_dom"]), kept_masks(S["std_mcq"]), kept_masks(S["std_text"])
print(f"kept-set Jaccard circuit vs standard {mean_jaccard(kc, ks):.3f} | circuit vs std_text {mean_jaccard(kc, kt):.3f} "
      f"| two random sets would overlap about {keep_frac / (2 - keep_frac):.3f}")
print(f"Spearman (mean over layers): circuit vs standard {mean_spearman(S['attr_dom'], S['std_mcq']):.3f} | "
      f"circuit vs std_text {mean_spearman(S['attr_dom'], S['std_text']):.3f} | "
      f"domain vs general attribution {mean_spearman(S['attr_dom'], S['attr_gen']):.3f}")

circ = np.array(res[f"circuit_guided|{chosen}"]["correct"])
std = np.array(res[f"standard_importance|{chosen}"]["correct"])
rnd = np.mean([res[f"random|{chosen}|{s}"]["correct"] for s in seeds], axis=0)
for name, other in [("standard", std), ("random (mean of seeds)", rnd)]:
    d, (lo, hi) = utils.paired_bootstrap_diff(circ, other, seed=frozen["config"]["seeds"]["bootstrap"])
    print(f"circuit minus {name} at {chosen:.0%}: {d * 100:+.1f} pts (paired 95% CI {lo * 100:+.1f} to {hi * 100:+.1f}) on n={len(circ)}")
print("These are pre-recovery development numbers on dev_eval, not the claim. The claim comes from the sealed test after recovery.")

unpruned dev_eval accuracy 0.596; rule threshold 0.477 (median of the three arms)
 reduction  random_mean  random_min  random_max  standard  circuit  std_text(ref)  ppl_random  ppl_standard  ppl_circuit  median_of_arms  passes_rule
      0.10        0.575       0.540       0.612     0.612    0.596          0.612      21.334        18.645       19.135           0.596         True
      0.25        0.392       0.316       0.468     0.588    0.572          0.540      43.499        27.639       34.349           0.572         True
      0.40        0.247       0.220       0.296     0.528    0.520          0.512    2979.298        56.219       81.219           0.520         True
      0.55        0.273       0.264       0.288     0.512    0.444          0.316  172477.893       179.677      273.901           0.444        False

[budget] chosen MLP neuron reduction 40% -> 974M params removed (29.1% of the model). Recorded in stage3_budget_choice.json.
kept-set Jaccard circuit vs standard 0.575

In [34]:
import os
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError

def get_write_token():
    tok = os.environ.get("HF_WRITE_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
        except Exception:
            tok = None
    return tok

def backup_to_hf(root=ROOT, repo_name="specialist-backup"):
    tok = get_write_token()
    if not tok:
        raise RuntimeError("No HF_WRITE_TOKEN secret found. Add it in Kaggle (Add-ons > Secrets) and attach it to this notebook.")
    api = HfApi(token=tok)
    who = api.whoami()
    auth = who.get("auth", {}).get("accessToken", {})
    print(f"[backup] user: {who['name']} | token role: {auth.get('role')} | name: {auth.get('displayName')}")
    repo_id = f"{who['name']}/{repo_name}"
    try:
        api.repo_info(repo_id, repo_type="dataset")
    except RepositoryNotFoundError:
        api.create_repo(repo_id=repo_id, repo_type="dataset", private=True)
    api.upload_folder(folder_path=str(root), repo_id=repo_id, repo_type="dataset",
                      ignore_patterns=["artifacts/*", "**/__pycache__/*"],
                      commit_message=f"backup {utils.now_utc()}")
    print(f"[backup] uploaded {root} to private dataset {repo_id}")

try:
    backup_to_hf()
except Exception as e:
    print(f"[backup] FAILED: {type(e).__name__}: {str(e)[:300]}")

[backup] user: Anirudh-Peddinti | token role: write | name: kaggle-write
[backup] uploaded /kaggle/working/specialist to private dataset Anirudh-Peddinti/specialist-backup


In [35]:
import json, time, importlib
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, utils
importlib.reload(scorekit)

frozen, rows, texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
SCORES = ROOT / "scores" / "neuron_scores.pt"
SWEEP = ROOT / "metrics" / "stage3_sweep.json"
OUT = ROOT / "metrics" / "stage3_sweep_offdomain.json"
for p in [SCORES, SWEEP, D / "gen_mcq_val.jsonl", D / "dom_text_val.jsonl"]:
    assert p.exists(), f"missing {p.name}: restore from the backup or re-run the earlier cell"
S = torch.load(SCORES)
sweep = json.loads(SWEEP.read_text())
L, Dm = S["attr_dom"].shape

gen_val = datakit.read_jsonl(D / "gen_mcq_val.jsonl")
gen_prompts = gen_val["question"].tolist()
gen_labels = gen_val["answer_idx"].to_numpy()
dom_val_text = datakit.read_jsonl(D / "dom_text_val.jsonl")["text"].tolist()[:64]

arm_scores = {"standard_importance": S["std_mcq"], "circuit_guided": S["attr_dom"], "standard_text": S["std_text"]}

# Rebuilds exactly the neuron sets used in Cell 3.4 (same scores, same seeds, same keep fractions)
def keep_for_key(key):
    if key == "base":
        return None
    parts = key.split("|")
    if parts[0] == "random":
        return scorekit.keep_indices(scorekit.random_scores(L, Dm, int(parts[2])), 1.0 - float(parts[1]))
    return scorekit.keep_indices(arm_scores[parts[0]], 1.0 - float(parts[1]))

tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

res = json.loads(OUT.read_text()) if OUT.exists() else {}
utils.free_gpu()
utils.assert_free_gpu(9, devices=[0])
model = scorekit.load_model(frozen["model_id"], 0)
masks = scorekit.NeuronMasks(model)

for key in sweep:
    if key in res:
        continue
    t = time.perf_counter()
    keep = keep_for_key(key)
    if keep is None:
        masks.reset()
    else:
        masks.set_keep(keep)
    lg = evalkit.score_mcq(model, tokenizer, gen_prompts, letter_ids, 8)
    acc, corr, hist = evalkit.accuracy_from_logits(lg, gen_labels)
    ppl = evalkit.corpus_ppl(model, tokenizer, dom_val_text)
    res[key] = {"gen_acc": acc, "gen_correct": corr.tolist(), "gen_pred_hist": hist, "dom_text_ppl": ppl}
    OUT.write_text(json.dumps(res))                      # saved after every evaluation, so a crash loses nothing
    print(f"{key:36s} off-domain acc {acc:.3f} | domain-text ppl {ppl:9.2f} | {time.perf_counter() - t:.0f}s")

masks.remove()
del model
utils.free_gpu()
utils.gpu_report("after off-domain sweep")
print("off-domain sweep complete (chance is about 25%; the most common answer letter in gen_mcq_val is about 27%)")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base                                 off-domain acc 0.590 | domain-text ppl     10.24 | 17s
random|0.1|0                         off-domain acc 0.535 | domain-text ppl     12.49 | 17s
random|0.1|1                         off-domain acc 0.485 | domain-text ppl     12.70 | 17s
random|0.1|2                         off-domain acc 0.530 | domain-text ppl     12.66 | 17s
standard_importance|0.1              off-domain acc 0.525 | domain-text ppl     11.12 | 17s
circuit_guided|0.1                   off-domain acc 0.565 | domain-text ppl     11.49 | 17s
standard_text|0.1                    off-domain acc 0.485 | domain-text ppl     10.65 | 17s
random|0.25|0                        off-domain acc 0.425 | domain-text ppl     20.62 | 17s
random|0.25|1                        off-domain acc 0.300 | domain-text ppl     21.60 | 17s
random|0.25|2                        off-domain acc 0.345 | domain-text ppl     23.67 | 17s
standard_importance|0.25             off-domain acc 0.460 | domain-text ppl     

In [36]:
import json
import numpy as np, pandas as pd
import utils

sweep = json.loads((ROOT / "metrics" / "stage3_sweep.json").read_text())
off = json.loads((ROOT / "metrics" / "stage3_sweep_offdomain.json").read_text())
bsel = json.loads((ROOT / "config" / "design_stage3_amendment.json").read_text())["amendment"]["budget_selection"]
chosen = json.loads((ROOT / "config" / "stage3_budget_choice.json").read_text())["chosen_mlp_neuron_reduction"]
grid, seeds = bsel["grid_mlp_neuron_reduction"], bsel["random_arm_seeds"]
missing = [k for k in sweep if k not in off]
assert not missing, f"off-domain results missing for {len(missing)} variants: re-run Cell 3.6"

def gmean(x):
    return float(np.exp(np.mean(np.log(x))))

base = {"dom_acc": sweep["base"]["acc"], "gen_acc": off["base"]["gen_acc"],
        "ppl_gen": sweep["base"]["ppl"], "ppl_dom": off["base"]["dom_text_ppl"]}
base_gap = base["dom_acc"] - base["gen_acc"]
print(f"unpruned: domain acc {base['dom_acc']:.3f} | off-domain acc {base['gen_acc']:.3f} | gap {base_gap:+.3f} | "
      f"ppl domain-text {base['ppl_dom']:.2f} / general-text {base['ppl_gen']:.2f}\n")

def keys_for(arm, r):
    return [f"random|{r}|{s}" for s in seeds] if arm == "random" else [f"{arm}|{r}"]

table = []
for r in grid:
    for arm in ["random", "standard_importance", "circuit_guided", "standard_text"]:
        ks = keys_for(arm, r)
        dom = np.mean([sweep[k]["acc"] for k in ks])
        gen = np.mean([off[k]["gen_acc"] for k in ks])
        pdm = gmean([off[k]["dom_text_ppl"] for k in ks])
        pgn = gmean([sweep[k]["ppl"] for k in ks])
        table.append({"reduction": r, "arm": arm, "dom_acc": dom, "gen_acc": gen,
                      "d_gap": (dom - gen) - base_gap,
                      "ppl_dom_x": pdm / base["ppl_dom"], "ppl_gen_x": pgn / base["ppl_gen"],
                      "ppl_spec": (pgn / base["ppl_gen"]) / (pdm / base["ppl_dom"])})
df = pd.DataFrame(table)
print("d_gap > 0 = pruning cost general ability more than domain ability; ppl_spec > 1 = general text degraded more than domain text")
print(df.round(3).to_string(index=False))

def arrays(arm, r):
    if arm == "random":
        d = np.mean([sweep[f"random|{r}|{s}"]["correct"] for s in seeds], axis=0)
        g = np.mean([off[f"random|{r}|{s}"]["gen_correct"] for s in seeds], axis=0)
    else:
        d = np.array(sweep[f"{arm}|{r}"]["correct"], dtype=float)
        g = np.array(off[f"{arm}|{r}"]["gen_correct"], dtype=float)
    return d, g

# Bootstraps domain questions and general questions separately; each is paired across variants
def gap_diff_ci(a, b, n_boot=5000, seed=0):
    (ad, ag), (bd, bg) = a, b
    rng = np.random.default_rng(seed)
    di = rng.integers(0, len(ad), (n_boot, len(ad)))
    gi = rng.integers(0, len(ag), (n_boot, len(ag)))
    stat = (ad[di].mean(1) - ag[gi].mean(1)) - (bd[di].mean(1) - bg[gi].mean(1))
    point = (ad.mean() - ag.mean()) - (bd.mean() - bg.mean())
    lo, hi = np.percentile(stat, [2.5, 97.5])
    return float(point * 100), float(lo * 100), float(hi * 100)

base_arr = (np.array(sweep["base"]["correct"], dtype=float), np.array(off["base"]["gen_correct"], dtype=float))
print(f"\nAt the chosen budget ({chosen:.0%}), change in the domain-minus-general gap, in points (95% CI):")
for arm in ["random", "standard_importance", "circuit_guided"]:
    p, lo, hi = gap_diff_ci(arrays(arm, chosen), base_arr)
    print(f"  {arm:20s} vs unpruned: {p:+6.1f}  ({lo:+.1f} to {hi:+.1f})")
for other in ["standard_importance", "random"]:
    p, lo, hi = gap_diff_ci(arrays("circuit_guided", chosen), arrays(other, chosen))
    print(f"  circuit_guided minus {other:20s}: {p:+6.1f}  ({lo:+.1f} to {hi:+.1f})")
print("Pre-recovery development numbers (dev_eval n=250, gen_mcq_val n=200). Not the claim.")

unpruned: domain acc 0.596 | off-domain acc 0.590 | gap +0.006 | ppl domain-text 10.24 / general-text 15.35

d_gap > 0 = pruning cost general ability more than domain ability; ppl_spec > 1 = general text degraded more than domain text
 reduction                 arm  dom_acc  gen_acc  d_gap  ppl_dom_x  ppl_gen_x  ppl_spec
      0.10              random    0.575    0.517  0.052      1.232      1.389     1.128
      0.10 standard_importance    0.612    0.525  0.081      1.085      1.215     1.120
      0.10      circuit_guided    0.596    0.565  0.025      1.121      1.247     1.112
      0.10       standard_text    0.612    0.485  0.121      1.040      1.220     1.173
      0.25              random    0.392    0.357  0.029      2.141      2.829     1.321
      0.25 standard_importance    0.588    0.460  0.122      1.289      1.801     1.397
      0.25      circuit_guided    0.572    0.520  0.046      1.454      2.238     1.539
      0.25       standard_text    0.540    0.465  0.069      

In [37]:
import json
import utils

try:
    ROOT
except NameError:
    raise RuntimeError("Run Cell 0.1 first: it defines ROOT and utils.")

a3 = json.loads((ROOT / "config" / "design_stage3_amendment.json").read_text())
choice = json.loads((ROOT / "config" / "stage3_budget_choice.json").read_text())
AMEND_B = ROOT / "config" / "design_stage3b_amendment.json"
FORCE_AMEND_B = False

AMENDMENT_B = {
    "parent_amendment_sha256": a3["amendment_sha256"],
    "status": "POST-HOC: defined after seeing the Stage 3 sweep (circuit_guided did not beat standard importance). "
              "Results for this variant are exploratory.",
    "variant": {
        "name": "circuit_ig",
        "method": "integrated gradients over a global gate path: every MLP neuron activation is scaled by alpha "
                  "(midpoints, K=5) from fully ablated (0) to intact (1), with the forward pass recomputed at each alpha; "
                  "score = mean over pairs of |mean_k sum_t dm/dgate|; same balanced pairs and metric as circuit_guided. "
                  "The first-order circuit_guided score is the alpha=1 endpoint only.",
    },
    "fixed": {"budget_mlp_neuron_reduction": choice["chosen_mlp_neuron_reduction"],
              "arm_for_the_claim": "unchanged: circuit_ig is reported alongside circuit_guided, never instead of it"},
    "reporting": "both circuit variants and the standard_text reference are reported; any win by circuit_ig alone is labelled post-hoc",
}
h = utils.sha256_json(AMENDMENT_B)
if AMEND_B.exists() and not FORCE_AMEND_B:
    old = json.loads(AMEND_B.read_text())
    if old["amendment_sha256"] != h:
        raise RuntimeError("Amendment B is already frozen and differs from your edits. Revert, or set FORCE_AMEND_B=True deliberately.")
    print(f"[amendment B] already frozen at {old['frozen_at']}, unchanged")
else:
    AMEND_B.write_text(json.dumps({"frozen_at": utils.now_utc(), "amendment_sha256": h, "amendment": AMENDMENT_B}, indent=2))
    print(f"[amendment B] wrote {AMEND_B.name} (sha {h[:12]}) before the IG variant has been scored")

[amendment B] wrote design_stage3b_amendment.json (sha b614c5e3abdd) before the IG variant has been scored


In [38]:
import json, time, importlib
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, utils

IG_SRC = r'''
import numpy as np
import torch

class GateHook:
    # Multiplies each MLP neuron activation (input of down_proj) by a per-position gate tensor
    def __init__(self, model):
        self.gates = {}
        self.handles = [layer.mlp.down_proj.register_forward_pre_hook(self._make(l))
                        for l, layer in enumerate(model.model.layers)]

    def _make(self, l):
        def pre(module, args):
            g = self.gates.get(l)
            if g is None:
                return None
            return (args[0] * g,)
        return pre

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []

def ig_pass(model, tokenizer, texts, letter_ids, contrast, steps=5, batch_size=2, max_len=512, scale=32.0):
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    L, Dm = model.config.num_hidden_layers, model.config.intermediate_size
    dev = model.get_input_embeddings().weight.device
    hook = GateHook(model)
    ids = torch.tensor(letter_ids, device=dev)
    A_abs = torch.zeros(L, Dm, device=dev)
    A_sgn = torch.zeros(L, Dm, device=dev)
    n_ex = 0
    order = np.argsort([len(t) for t in texts])
    try:
        for s in range(0, len(texts), batch_size):
            idx = order[s:s + batch_size]
            enc = tokenizer([texts[i] for i in idx], return_tensors="pt", padding=True,
                            truncation=True, max_length=max_len).to(dev)
            B, T = enc["input_ids"].shape
            v = enc["attention_mask"].to(torch.float32)[..., None]            # [B, T, 1]
            c = ids[torch.tensor([contrast[i][0] for i in idx], device=dev)]
            k = ids[torch.tensor([contrast[i][1] for i in idx], device=dev)]
            acc = torch.zeros(B, L, Dm, device=dev)
            for step in range(steps):
                alpha = (step + 0.5) / steps
                hook.gates = {l: torch.full((B, T, Dm), alpha, dtype=torch.float16, device=dev, requires_grad=True)
                              for l in range(L)}
                with torch.enable_grad():
                    try:
                        logits = model(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :]
                    except TypeError:
                        logits = model(**enc, use_cache=False).logits[:, -1, :]
                    rows_i = torch.arange(B, device=dev)
                    m = logits[rows_i, c].float() - logits[rows_i, k].float()
                    (m.sum() * scale).backward()
                for l in range(L):
                    acc[:, l] += (hook.gates[l].grad.float() * v).sum(dim=1)     # [B, Dm]
                hook.gates = {}
                del logits, m
            acc /= steps
            A_abs += acc.abs().sum(dim=0)
            A_sgn += acc.sum(dim=0)
            n_ex += B
    finally:
        hook.remove()
    return {"attr_abs_mean": (A_abs / max(n_ex, 1) / scale).cpu(),
            "attr_signed_mean": (A_sgn / max(n_ex, 1) / scale).cpu(), "n_ex": n_ex}
'''
(ROOT / "utils" / "scorekit_ig.py").write_text(IG_SRC)
import scorekit_ig
importlib.reload(scorekit_ig)

SMOKE = False     # True: the 8 LONGEST prompts (worst-case memory and time), nothing saved. Then False for the real run.
STEPS = 5

frozen, rows, texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
dom_pairs = datakit.read_jsonl(D / "dom_pairs_balanced.jsonl")
prompts = dom_pairs["question"].tolist()
contrast = [(int(a), int(b)) for a, b in zip(dom_pairs["answer_idx"], dom_pairs["corrupt_answer_idx"])]
if SMOKE:
    order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))[:8]
    prompts, contrast = [prompts[i] for i in order], [contrast[i] for i in order]

utils.free_gpu()
utils.assert_free_gpu(9, devices=[0])
model = scorekit.load_model(frozen["model_id"], 0)
torch.cuda.reset_peak_memory_stats(0)
t = time.perf_counter()
r = scorekit_ig.ig_pass(model, tokenizer, prompts, letter_ids, contrast, steps=STEPS, batch_size=2, max_len=512)
el = time.perf_counter() - t
peak = torch.cuda.max_memory_allocated(0) / 1024**3
print(f"IG pass: {r['n_ex']} prompts x {STEPS} steps in {el:.0f}s | peak GPU {peak:.2f} GiB"
      + ("   <-- above 13 GiB: use batch_size=1" if peak > 13.0 else ""))
if SMOKE:
    print(f"estimated full run (192 prompts): about {el / r['n_ex'] * 192 / 60:.0f} minutes (longest prompts, so likely an overestimate)")

assert torch.isfinite(r["attr_abs_mean"]).all(), "NaN or inf in IG scores"
assert r["attr_abs_mean"].sum() > 0, "IG attribution is all zero: gradients are not reaching the gates"
print("attr_ig mean per layer (every 5th):",
      {l: round(float(r["attr_abs_mean"][l].mean()), 5) for l in range(0, r["attr_abs_mean"].shape[0], 5)})
print(f"neurons with exactly zero attribution: {(r['attr_abs_mean'] == 0).float().mean().item():.4%}")

if SMOKE:
    print("SMOKE run: nothing saved. If peak memory and time look fine, set SMOKE = False and re-run.")
else:
    S = torch.load(ROOT / "scores" / "neuron_scores.pt")
    from scipy.stats import spearmanr
    rho = np.mean([spearmanr(r["attr_abs_mean"][l].numpy(), S["attr_dom"][l].numpy()).correlation
                   for l in range(r["attr_abs_mean"].shape[0])])
    print(f"Spearman (mean over layers) circuit_ig vs first-order circuit_guided: {rho:.3f}")
    torch.save({"attr_ig": r["attr_abs_mean"], "attr_ig_signed": r["attr_signed_mean"],
                "meta": {"smoke": False, "steps": STEPS, "n_dom": r["n_ex"], "created_at": utils.now_utc()}},
               ROOT / "scores" / "neuron_scores_ig.pt")
    print("saved scores/neuron_scores_ig.pt")

del model
utils.free_gpu()
utils.gpu_report("after IG scoring")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

IG pass: 192 prompts x 5 steps in 99s | peak GPU 8.62 GiB
attr_ig mean per layer (every 5th): {0: 0.00519, 5: 0.00823, 10: 0.0049, 15: 0.00368, 20: 0.00228, 25: 0.00142, 30: 0.00166, 35: 0.00194}
neurons with exactly zero attribution: 0.0000%
Spearman (mean over layers) circuit_ig vs first-order circuit_guided: 0.524
saved scores/neuron_scores_ig.pt
[gpu 0 after IG scoring] free 14.38 / 14.56 GiB | allocated 0.02 | reserved 0.04
[gpu 1 after IG scoring] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00


{0: {'free_gib': 14.38165283203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.016607284545898438,
  'reserved_gib': 0.041015625},
 1: {'free_gib': 14.45977783203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0,
  'reserved_gib': 0.0}}

In [39]:
import json, time, importlib
import numpy as np, pandas as pd, torch
from scipy.stats import spearmanr
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, utils
importlib.reload(scorekit)

frozen, rows, texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
IGP = ROOT / "scores" / "neuron_scores_ig.pt"
assert IGP.exists(), "neuron_scores_ig.pt not found: run Cell 3.9 with SMOKE=False first"
IG = torch.load(IGP)
assert not IG["meta"]["smoke"], "IG scores came from a smoke run"
S = torch.load(ROOT / "scores" / "neuron_scores.pt")
sweep = json.loads((ROOT / "metrics" / "stage3_sweep.json").read_text())
off = json.loads((ROOT / "metrics" / "stage3_sweep_offdomain.json").read_text())
bsel = json.loads((ROOT / "config" / "design_stage3_amendment.json").read_text())["amendment"]["budget_selection"]
chosen = json.loads((ROOT / "config" / "stage3_budget_choice.json").read_text())["chosen_mlp_neuron_reduction"]
grid, seeds = bsel["grid_mlp_neuron_reduction"], bsel["random_arm_seeds"]

gen_val = datakit.read_jsonl(D / "gen_mcq_val.jsonl")
gen_prompts, gen_labels = gen_val["question"].tolist(), gen_val["answer_idx"].to_numpy()
dom_val_text = datakit.read_jsonl(D / "dom_text_val.jsonl")["text"].tolist()[:64]

tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
OUT = ROOT / "metrics" / "stage3b_ig_sweep.json"
res = json.loads(OUT.read_text()) if OUT.exists() else {}

utils.free_gpu()
utils.assert_free_gpu(9, devices=[0])
model = scorekit.load_model(frozen["model_id"], 0)
masks = scorekit.NeuronMasks(model)
for r in grid:
    key = f"circuit_ig|{r}"
    if key in res:
        continue
    t = time.perf_counter()
    keep = scorekit.keep_indices(IG["attr_ig"], 1.0 - r)
    d = scorekit.eval_keep(model, masks, tokenizer, rows, texts, letter_ids, keep)      # sets the masks and scores dev_eval
    lg = evalkit.score_mcq(model, tokenizer, gen_prompts, letter_ids, 8)
    gacc, gcorr, _ = evalkit.accuracy_from_logits(lg, gen_labels)
    res[key] = {"acc": d["acc"], "correct": d["correct"], "ppl": d["ppl"], "gen_acc": gacc,
                "gen_correct": gcorr.tolist(), "dom_text_ppl": evalkit.corpus_ppl(model, tokenizer, dom_val_text)}
    OUT.write_text(json.dumps(res))
    print(f"{key:20s} dom acc {d['acc']:.3f} | off-domain acc {gacc:.3f} | {time.perf_counter() - t:.0f}s")
masks.remove()
del model
utils.free_gpu()

# ---------------- comparison of all arms ----------------
CH = 0.25
b_dom, b_gen = sweep["base"]["acc"], off["base"]["gen_acc"]
def skill(acc, base):
    return (acc - CH) / (base - CH)
def gmean(x):
    return float(np.exp(np.mean(np.log(x))))

def get(arm, r):
    if arm == "random":
        ks = [f"random|{r}|{s}" for s in seeds]
        d = np.mean([sweep[k]["correct"] for k in ks], axis=0)
        g = np.mean([off[k]["gen_correct"] for k in ks], axis=0)
        return d, g, gmean([off[k]["dom_text_ppl"] for k in ks]), gmean([sweep[k]["ppl"] for k in ks])
    if arm == "circuit_ig":
        x = res[f"circuit_ig|{r}"]
        return np.array(x["correct"], float), np.array(x["gen_correct"], float), x["dom_text_ppl"], x["ppl"]
    k = f"{arm}|{r}"
    return np.array(sweep[k]["correct"], float), np.array(off[k]["gen_correct"], float), off[k]["dom_text_ppl"], sweep[k]["ppl"]

arms = ["random", "standard_importance", "circuit_guided", "circuit_ig", "standard_text"]
table = []
for r in grid:
    for arm in arms:
        d, g, pdm, pgn = get(arm, r)
        table.append({"reduction": r, "arm": arm, "dom_acc": d.mean(), "gen_acc": g.mean(),
                      "dom_skill": skill(d.mean(), b_dom), "gen_skill": skill(g.mean(), b_gen),
                      "ppl_dom_x": pdm / off["base"]["dom_text_ppl"], "ppl_gen_x": pgn / sweep["base"]["ppl"]})
print("skill = (accuracy - 0.25) / (unpruned accuracy - 0.25); specialization = domain skill kept while general skill drops")
print(pd.DataFrame(table).round(3).to_string(index=False))

def gap_diff_ci(a, b, n_boot=5000, seed=0):
    (ad, ag), (bd, bg) = a, b
    rng = np.random.default_rng(seed)
    di = rng.integers(0, len(ad), (n_boot, len(ad)))
    gi = rng.integers(0, len(ag), (n_boot, len(ag)))
    stat = (ad[di].mean(1) - ag[gi].mean(1)) - (bd[di].mean(1) - bg[gi].mean(1))
    return float(((ad.mean() - ag.mean()) - (bd.mean() - bg.mean())) * 100), *[float(x) * 100 for x in np.percentile(stat, [2.5, 97.5])]

base_arr = (np.array(sweep["base"]["correct"], float), np.array(off["base"]["gen_correct"], float))
ig_arr = get("circuit_ig", chosen)[:2]
print(f"\nAt {chosen:.0%}: circuit_ig change in domain-minus-general gap vs unpruned: "
      "{:+.1f} pts (95% CI {:+.1f} to {:+.1f})".format(*gap_diff_ci(ig_arr, base_arr)))
for other in ["standard_importance", "circuit_guided", "standard_text"]:
    print(f"circuit_ig minus {other:20s}: gap {{:+.1f}} pts ({{:+.1f}} to {{:+.1f}})".format(*gap_diff_ci(ig_arr, get(other, chosen)[:2])))
    d1, d2 = ig_arr[0], get(other, chosen)[0]
    p, (lo, hi) = utils.paired_bootstrap_diff(d1, d2, seed=frozen["config"]["seeds"]["bootstrap"])
    print(f"{'':24s} domain accuracy {p * 100:+.1f} pts ({lo * 100:+.1f} to {hi * 100:+.1f})")

keep_frac = 1.0 - chosen
def kept(sc):
    out = []
    for idx in scorekit.keep_indices(sc, keep_frac):
        m = torch.zeros(sc.shape[1], dtype=torch.bool)
        m[idx] = True
        out.append(m)
    return out
ka, kb = kept(IG["attr_ig"]), kept(S["attr_dom"])
jac = np.mean([(x & y).sum().item() / (x | y).sum().item() for x, y in zip(ka, kb)])
print(f"kept-set Jaccard circuit_ig vs circuit_guided {jac:.3f} (random sets would overlap about {keep_frac / (2 - keep_frac):.3f})")
print("Post-hoc and exploratory: dev_eval n=250, gen_mcq_val n=200. Not the claim.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

circuit_ig|0.1       dom acc 0.588 | off-domain acc 0.575 | 33s
circuit_ig|0.25      dom acc 0.544 | off-domain acc 0.505 | 33s
circuit_ig|0.4       dom acc 0.536 | off-domain acc 0.420 | 32s
circuit_ig|0.55      dom acc 0.440 | off-domain acc 0.325 | 32s
skill = (accuracy - 0.25) / (unpruned accuracy - 0.25); specialization = domain skill kept while general skill drops
 reduction                 arm  dom_acc  gen_acc  dom_skill  gen_skill  ppl_dom_x  ppl_gen_x
      0.10              random    0.575    0.517      0.938      0.784      1.232      1.389
      0.10 standard_importance    0.612    0.525      1.046      0.809      1.085      1.215
      0.10      circuit_guided    0.596    0.565      1.000      0.926      1.121      1.247
      0.10          circuit_ig    0.588    0.575      0.977      0.956      1.176      1.365
      0.10       standard_text    0.612    0.485      1.046      0.691      1.040      1.220
      0.25              random    0.392    0.357      0.410      0.31

## Stage 4

In [40]:
import json
import utils

try:
    ROOT
except NameError:
    raise RuntimeError("Run Cell 0.1 first: it defines ROOT and utils.")

a3 = json.loads((ROOT / "config" / "design_stage3_amendment.json").read_text())
choice = json.loads((ROOT / "config" / "stage3_budget_choice.json").read_text())
AMEND_C = ROOT / "config" / "design_stage4_amendment.json"
FORCE_AMEND_C = False

ARMS = ["random", "standard_importance", "circuit_guided", "standard_text"]   # add "circuit_ig" if time allows

AMENDMENT_C = {
    "parent_amendment_sha256": a3["amendment_sha256"],
    "budget_mlp_neuron_reduction": choice["chosen_mlp_neuron_reduction"],
    "arms_to_recover": ARMS,
    "claim_arms": ["random", "standard_importance", "circuit_guided"],
    "reference_arms": [a for a in ARMS if a not in ("random", "standard_importance", "circuit_guided")],
    "reference_arm_note": "post-hoc additions, reported alongside and labelled as such",
    "random_arm_recovery_seed": 0,
    "student": "base model with MLP width physically reduced to the same k in every layer",
    "recovery_spec": "frozen in its own amendment before any recovery run; identical for every arm; no gold labels",
    "pre_recovery_findings_on_dev_eval": "circuit_guided and the post-hoc circuit_ig were statistically tied with "
                                         "standard_importance on domain accuracy and on the domain-minus-general gap at 40%; "
                                         "all informed arms beat random by a wide margin.",
}
h = utils.sha256_json(AMENDMENT_C)
if AMEND_C.exists() and not FORCE_AMEND_C:
    old = json.loads(AMEND_C.read_text())
    if old["amendment_sha256"] != h:
        raise RuntimeError("Amendment C is already frozen and differs from your edits. Revert, or set FORCE_AMEND_C=True deliberately.")
    print(f"[amendment C] already frozen at {old['frozen_at']}, unchanged")
else:
    AMEND_C.write_text(json.dumps({"frozen_at": utils.now_utc(), "amendment_sha256": h, "amendment": AMENDMENT_C}, indent=2))
    print(f"[amendment C] wrote {AMEND_C.name} (sha {h[:12]}); arms: {ARMS}")

[amendment C] wrote design_stage4_amendment.json (sha 25c68d3367cd); arms: ['random', 'standard_importance', 'circuit_guided', 'standard_text']


In [41]:
import json, importlib
import torch
import scorekit, utils

PRUNEKIT_SRC = r'''
import torch
import torch.nn as nn

# Physically removes MLP neurons: gate/up rows and down columns. Every layer must keep the same number.
def prune_mlp_width(model, keep):
    k = len(keep[0])
    assert all(len(x) == k for x in keep), "keep sets must have the same size in every layer"
    for l, layer in enumerate(model.model.layers):
        mlp = layer.mlp
        idx = keep[l].to(mlp.gate_proj.weight.device)
        for name in ("gate_proj", "up_proj"):
            lin = getattr(mlp, name)
            assert lin.bias is None, "bias terms are not handled"
            lin.weight = nn.Parameter(lin.weight.data[idx].contiguous(), requires_grad=False)
            lin.out_features = k
        d = mlp.down_proj
        assert d.bias is None, "bias terms are not handled"
        d.weight = nn.Parameter(d.weight.data[:, idx].contiguous(), requires_grad=False)
        d.in_features = k
        if hasattr(mlp, "intermediate_size"):
            mlp.intermediate_size = k
    model.config.intermediate_size = k
    torch.cuda.empty_cache()
    return model
'''
(ROOT / "utils" / "prunekit.py").write_text(PRUNEKIT_SRC)
import prunekit
importlib.reload(prunekit)

amend = json.loads((ROOT / "config" / "design_stage4_amendment.json").read_text())["amendment"]
chosen = amend["budget_mlp_neuron_reduction"]
S = torch.load(ROOT / "scores" / "neuron_scores.pt")
L, Dm = S["attr_dom"].shape

score_for = {"standard_importance": S["std_mcq"], "circuit_guided": S["attr_dom"], "standard_text": S["std_text"]}
if "circuit_ig" in amend["arms_to_recover"]:
    score_for["circuit_ig"] = torch.load(ROOT / "scores" / "neuron_scores_ig.pt")["attr_ig"]
score_for["random"] = scorekit.random_scores(L, Dm, amend["random_arm_recovery_seed"])

keeps = {arm: scorekit.keep_indices(score_for[arm], 1.0 - chosen) for arm in amend["arms_to_recover"]}
k = len(next(iter(keeps.values()))[0])
removed = (Dm - k) * 3 * 2048 * L
torch.save({"keeps": keeps, "budget": chosen, "k": k, "created_at": utils.now_utc()}, ROOT / "scores" / "keep_indices.pt")
print(f"arms: {list(keeps)} | each layer keeps k={k} of {Dm} neurons | removes {removed:,} parameters "
      f"({100 * removed / 3_349_227_520:.1f}% of the model)")
print("saved scores/keep_indices.pt")

arms: ['random', 'standard_importance', 'circuit_guided', 'standard_text'] | each layer keeps k=6605 of 11008 neurons | removes 973,873,152 parameters (29.1% of the model)
saved scores/keep_indices.pt


In [42]:
import copy, importlib
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import evalkit, scorekit, prunekit, utils
importlib.reload(prunekit)

frozen, rows, texts, letter_ids = evalkit.load_eval_inputs(ROOT)
K = torch.load(ROOT / "scores" / "keep_indices.pt")
ARM = "circuit_guided"
keep, k = K["keeps"][ARM], K["k"]

tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
prompts, labels = evalkit.build_prompts(rows[:16], "primary", 0)

utils.free_gpu()
utils.assert_free_gpu(9, devices=[0])
model = scorekit.load_model(frozen["model_id"], 0)
n_before = sum(p.numel() for p in model.parameters())

# 1) masked model (the thing the sweep measured)
masks = scorekit.NeuronMasks(model)
masks.set_keep(keep)
lg_masked = evalkit.score_mcq(model, tokenizer, prompts, letter_ids, 8)
masks.remove()

# 2) physically pruned model
prunekit.prune_mlp_width(model, keep)
n_after = sum(p.numel() for p in model.parameters())
lg_pruned = evalkit.score_mcq(model, tokenizer, prompts, letter_ids, 8)

diff = float(np.abs(lg_masked - lg_pruned).max())
agree = float((lg_masked.argmax(1) == lg_pruned.argmax(1)).mean())
expected_removed = (11008 - k) * 3 * 2048 * 36
print(f"parameters {n_before:,} -> {n_after:,} (removed {n_before - n_after:,}; expected {expected_removed:,})")
print(f"max |logit difference| masked vs pruned: {diff:.4f} | argmax agreement on 16 prompts: {agree:.2f}")
print(f"pruned model weights: {model.get_memory_footprint() / 1024**3:.2f} GiB (fp16)")
assert n_before - n_after == expected_removed, "parameter count does not match the plan"
assert agree >= 15 / 16 and diff < 0.5, "masked and pruned models disagree: do not proceed (thresholds are my guess)"

# 3) a stock config with intermediate_size=k must reproduce this architecture exactly (no disk needed)
try:
    cfg = copy.deepcopy(model.config)
    cfg.intermediate_size = k
    with torch.device("meta"):
        ref = AutoModelForCausalLM.from_config(cfg)
    ref_shapes = {n: tuple(p.shape) for n, p in ref.named_parameters()}
    mod_shapes = {n: tuple(p.shape) for n, p in model.named_parameters()}
    assert ref_shapes == mod_shapes, "pruned shapes differ from a stock config with intermediate_size=k"
    print(f"stock config with intermediate_size={k} reproduces the pruned architecture exactly")
except Exception as e:
    print(f"[warn] meta-device shape check could not run: {type(e).__name__}: {str(e)[:200]}")

del model
utils.free_gpu()
utils.gpu_report("after Stage 4 check")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

parameters 3,349,227,520 -> 2,375,354,368 (removed 973,873,152; expected 973,873,152)
max |logit difference| masked vs pruned: 0.0156 | argmax agreement on 16 prompts: 1.00
pruned model weights: 4.42 GiB (fp16)
stock config with intermediate_size=6605 reproduces the pruned architecture exactly
[gpu 0 after Stage 4 check] free 14.38 / 14.56 GiB | allocated 0.02 | reserved 0.04
[gpu 1 after Stage 4 check] free 14.46 / 14.56 GiB | allocated 0.00 | reserved 0.00


{0: {'free_gib': 14.38165283203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.016607284545898438,
  'reserved_gib': 0.041015625},
 1: {'free_gib': 14.45977783203125,
  'total_gib': 14.56219482421875,
  'allocated_gib': 0.0,
  'reserved_gib': 0.0}}

In [43]:
import os
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError

def get_write_token():
    tok = os.environ.get("HF_WRITE_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
        except Exception:
            tok = None
    return tok

def backup_to_hf(root=ROOT, repo_name="specialist-backup"):
    tok = get_write_token()
    if not tok:
        raise RuntimeError("No HF_WRITE_TOKEN secret found. Add it in Kaggle (Add-ons > Secrets) and attach it to this notebook.")
    api = HfApi(token=tok)
    who = api.whoami()
    auth = who.get("auth", {}).get("accessToken", {})
    print(f"[backup] user: {who['name']} | token role: {auth.get('role')} | name: {auth.get('displayName')}")
    repo_id = f"{who['name']}/{repo_name}"
    try:
        api.repo_info(repo_id, repo_type="dataset")
    except RepositoryNotFoundError:
        api.create_repo(repo_id=repo_id, repo_type="dataset", private=True)
    api.upload_folder(folder_path=str(root), repo_id=repo_id, repo_type="dataset",
                      ignore_patterns=["artifacts/*", "**/__pycache__/*"],
                      commit_message=f"backup {utils.now_utc()}")
    print(f"[backup] uploaded {root} to private dataset {repo_id}")

try:
    backup_to_hf()
except Exception as e:
    print(f"[backup] FAILED: {type(e).__name__}: {str(e)[:300]}")

[backup] user: Anirudh-Peddinti | token role: write | name: kaggle-write
[backup] uploaded /kaggle/working/specialist to private dataset Anirudh-Peddinti/specialist-backup


## Stage 5

In [49]:
import sys, subprocess, importlib.util
r = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True, text=True)
print((r.stdout or r.stderr).strip()[-200:])
if importlib.util.find_spec("peft") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], capture_output=True, text=True)
    print("installed peft" if r.returncode == 0 else r.stderr[-300:])
import peft
print("peft", peft.__version__)

peft 0.21.2


In [50]:
import sys, importlib
from pathlib import Path

try:
    ROOT
except NameError:
    raise RuntimeError("Run Cell 0.1 first: it defines ROOT and utils.")
UTILS = ROOT / "utils"
if str(UTILS) not in sys.path:
    sys.path.insert(0, str(UTILS))

missing = [f for f in ["utils.py", "evalkit.py", "datakit.py", "scorekit.py", "prunekit.py"] if not (UTILS / f).exists()]
if missing:
    raise RuntimeError(f"missing in {UTILS}: {missing}. Run the cells that write them: "
                       "utils (0.1), evalkit (1.1), datakit (2.1), scorekit (3.2), prunekit (4.2).")
needed = ["scores/keep_indices.pt", "config/design_stage4_amendment.json", "data/dom_text_train.jsonl",
          "data/dom_mcq_train_nolabels.jsonl", "data/gen_mcq_val.jsonl", "data/dom_text_val.jsonl"]
gone = [f for f in needed if not (ROOT / f).exists()]
if gone:
    raise RuntimeError(f"missing data or config files: {gone}. Re-run the earlier stage that writes them, or restore from the backup.")
try:
    import peft
    print("peft", peft.__version__)
except ImportError:
    raise RuntimeError("peft is not installed: run Cell 5.0 (or Cell 0.0) first.")

RECOVERYKIT_SRC = r'''
import gc, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import evalkit, scorekit, prunekit, utils

DEFAULT_SPEC = {
    "lora": {"r": 16, "alpha": 32,
             "targets": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]},
    "lr": 2e-4, "warmup_steps": 50, "min_lr_frac": 0.1, "weight_decay": 0.0, "grad_clip": 1.0,
    "block": 256, "batch_blocks": 4, "positions_per_block": 64,
    "mcq_batch": 2, "mcq_weight": 1.0,
    "teacher": "unpruned base, fp16, on GPU 1",
    "uses_gold_labels": False,
}

# Replaces every parameter's storage with an empty tensor, so GPU memory is freed even if a stored
# traceback still references the model object.
def release(*models):
    for m in models:
        if m is None:
            continue
        for p in m.parameters():
            p.data = torch.empty(0, device=p.device, dtype=p.dtype)
        for b in m.buffers():
            b.data = torch.empty(0, device=b.device, dtype=b.dtype)
    gc.collect()
    torch.cuda.empty_cache()

# Packs the token stream into fixed-length blocks, each starting with BOS
def build_blocks(tokenizer, texts, block=256):
    bos, eos = tokenizer.bos_token_id, tokenizer.eos_token_id
    stream = []
    for ids in tokenizer(texts, add_special_tokens=False)["input_ids"]:
        stream.extend(ids)
        if eos is not None:
            stream.append(eos)
    body = block - (1 if bos is not None else 0)
    n = len(stream) // body
    arr = np.array(stream[: n * body], dtype=np.int64).reshape(n, body)
    if bos is not None:
        arr = np.concatenate([np.full((n, 1), bos, dtype=np.int64), arr], axis=1)
    return torch.from_numpy(arr)

def lr_at(step, steps, spec):
    if step < spec["warmup_steps"]:
        return spec["lr"] * (step + 1) / spec["warmup_steps"]
    prog = (step - spec["warmup_steps"]) / max(1, steps - spec["warmup_steps"])
    return spec["lr"] * (spec["min_lr_frac"] + (1 - spec["min_lr_frac"]) * 0.5 * (1 + math.cos(math.pi * prog)))

# Stateless sampling: the batch at a given step depends only on (seed, step), so resuming is exact
def sample_step(step, n_blocks, n_mcq, spec, seed):
    g = torch.Generator().manual_seed(seed * 100003 + step)
    bi = torch.randperm(n_blocks, generator=g)[: spec["batch_blocks"]]
    mi = torch.randperm(n_mcq, generator=g)[: spec["mcq_batch"]]
    pos = torch.randperm(spec["block"], generator=g)[: spec["positions_per_block"]].sort().values
    return bi, mi, pos

def pos_logits(model, ids, pos, dev):
    return model(input_ids=ids.to(dev), use_cache=False, logits_to_keep=pos.to(dev)).logits   # [B, P, V]

def mcq_letter_logits(model, tokenizer, prompts, letter_ids, dev, max_len=512):
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(dev)
    out = model(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :]
    return out.float().index_select(-1, torch.tensor(letter_ids, device=out.device))

def kl_div(student_logits, teacher_logits):
    sl = F.log_softmax(student_logits.float(), dim=-1)
    tl = F.log_softmax(teacher_logits.to(sl.device).float(), dim=-1)
    return (tl.exp() * (tl - sl)).sum(-1).mean()

def build_student(model_id, keep, device, spec):
    from peft import LoraConfig, get_peft_model
    m = scorekit.load_model(model_id, device)
    try:
        prunekit.prune_mlp_width(m, keep)
        m.config.use_cache = False
        m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        m.enable_input_require_grads()
        cfg = LoraConfig(r=spec["lora"]["r"], lora_alpha=spec["lora"]["alpha"], lora_dropout=0.0,
                         target_modules=spec["lora"]["targets"], task_type="CAUSAL_LM")
        pm = get_peft_model(m, cfg)
        pm.train()
        return pm
    except BaseException:
        release(m)
        raise

def train(student, teacher, tokenizer, blocks, mcq_prompts, letter_ids, spec, steps, seed,
          ckpt_path=None, ckpt_every=250, log_every=10):
    s_dev = student.get_input_embeddings().weight.device
    t_dev = teacher.get_input_embeddings().weight.device
    params = [p for p in student.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=spec["lr"], weight_decay=spec["weight_decay"])
    scaler = torch.amp.GradScaler("cuda")
    log = {"loss": [], "kl_text": [], "kl_mcq": [], "step_s": []}
    start = 0
    if ckpt_path is not None and Path(ckpt_path).exists():
        ck = torch.load(ckpt_path, map_location="cpu")
        named = dict(student.named_parameters())
        for n, v in ck["lora"].items():
            named[n].data.copy_(v.to(named[n].device))
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        log, start = ck["log"], ck["step"]
        print(f"[resume] continuing from step {start}")
    student.train()
    for step in range(start, steps):
        t0 = time.perf_counter()
        bi, mi, pos = sample_step(step, len(blocks), len(mcq_prompts), spec, seed)
        ids = blocks[bi]
        mp = [mcq_prompts[i] for i in mi.tolist()]
        with torch.no_grad():
            t_text = pos_logits(teacher, ids, pos, t_dev)
            t_mcq = mcq_letter_logits(teacher, tokenizer, mp, letter_ids, t_dev)
        s_text = pos_logits(student, ids, pos, s_dev)
        s_mcq = mcq_letter_logits(student, tokenizer, mp, letter_ids, s_dev)
        kl_t, kl_m = kl_div(s_text, t_text), kl_div(s_mcq, t_mcq)
        loss = kl_t + spec["mcq_weight"] * kl_m
        for g in opt.param_groups:
            g["lr"] = lr_at(step, steps, spec)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, spec["grad_clip"])
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        for d in {s_dev, t_dev}:
            torch.cuda.synchronize(d)
        log["step_s"].append(time.perf_counter() - t0)
        if step % log_every == 0 or step == steps - 1:
            lv, kt, km = float(loss.detach()), float(kl_t.detach()), float(kl_m.detach())
            log["loss"].append((step, lv))
            log["kl_text"].append((step, kt))
            log["kl_mcq"].append((step, km))
            print(f"step {step:5d}/{steps} | loss {lv:.4f} | kl_text {kt:.4f} | kl_mcq {km:.4f} "
                  f"| {log['step_s'][-1]:.2f}s/step", flush=True)
        if ckpt_path is not None and (step + 1) % ckpt_every == 0:
            torch.save({"step": step + 1, "log": log, "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "lora": {n: p.detach().cpu() for n, p in student.named_parameters() if p.requires_grad}},
                       ckpt_path)
    lora_state = {n: p.detach().cpu() for n, p in student.named_parameters() if p.requires_grad}
    return log, lora_state

def eval_all(model, tokenizer, rows, gen_prompts, gen_labels, wiki_texts, dom_texts, letter_ids):
    model.eval()
    prompts, labels = evalkit.build_prompts(rows, "primary", 0)
    dacc, dcorr, _ = evalkit.accuracy_from_logits(evalkit.score_mcq(model, tokenizer, prompts, letter_ids, 8), labels)
    gacc, gcorr, _ = evalkit.accuracy_from_logits(evalkit.score_mcq(model, tokenizer, gen_prompts, letter_ids, 8), gen_labels)
    return {"dom_acc": dacc, "dom_correct": dcorr.tolist(), "gen_acc": gacc, "gen_correct": gcorr.tolist(),
            "ppl_wiki": evalkit.corpus_ppl(model, tokenizer, wiki_texts),
            "ppl_dom": evalkit.corpus_ppl(model, tokenizer, dom_texts)}
'''
(UTILS / "recoverykit.py").write_text(RECOVERYKIT_SRC)
importlib.invalidate_caches()
import recoverykit
importlib.reload(recoverykit)
print("recoverykit written and imported; default spec:", recoverykit.DEFAULT_SPEC)

peft 0.21.2
recoverykit written and imported; default spec: {'lora': {'r': 16, 'alpha': 32, 'targets': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']}, 'lr': 0.0002, 'warmup_steps': 50, 'min_lr_frac': 0.1, 'weight_decay': 0.0, 'grad_clip': 1.0, 'block': 256, 'batch_blocks': 4, 'positions_per_block': 64, 'mcq_batch': 2, 'mcq_weight': 1.0, 'teacher': 'unpruned base, fp16, on GPU 1', 'uses_gold_labels': False}


In [51]:
import json, time, importlib
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, recoverykit, utils
importlib.reload(recoverykit)

PROBE_STEPS = 20
frozen, rows, wiki_texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
spec = recoverykit.DEFAULT_SPEC
tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = datakit.read_jsonl(D / "dom_text_train.jsonl")["text"].tolist()
blocks = recoverykit.build_blocks(tokenizer, texts, spec["block"])
mcq_prompts = datakit.read_jsonl(D / "dom_mcq_train_nolabels.jsonl")["question"].tolist()
K = torch.load(ROOT / "scores" / "keep_indices.pt")
print(f"{len(blocks):,} blocks of {spec['block']} tokens = {blocks.numel() / 1e6:.2f}M tokens | {len(mcq_prompts)} label-free MCQ prompts")

utils.free_gpu()
utils.assert_free_gpu(9, devices=[0])
utils.assert_free_gpu(7, devices=[1])
teacher = student = None
try:
    teacher = scorekit.load_model(frozen["model_id"], 1)
    teacher.eval()
    student = recoverykit.build_student(frozen["model_id"], K["keeps"]["circuit_guided"], 0, spec)
    print(f"trainable LoRA parameters: {sum(p.numel() for p in student.parameters() if p.requires_grad) / 1e6:.1f}M")
    for d in (0, 1):
        torch.cuda.reset_peak_memory_stats(d)
    log = recoverykit.train(student, teacher, tokenizer, blocks, mcq_prompts, letter_ids, spec,
                            steps=PROBE_STEPS, seed=frozen["config"]["seeds"]["train"], ckpt_path=None, log_every=5)[0]
    med = float(np.median(log["step_s"][3:]))
    print(f"\nmedian step time {med:.2f}s (first 3 steps excluded)")
    for d in (0, 1):
        print(f"peak GPU {d}: {torch.cuda.max_memory_allocated(d) / 1024**3:.2f} GiB")
    for steps in (500, 1000, 1500, 2000):
        print(f"{steps:5d} steps: {steps * med / 60:5.0f} min per arm | {steps * med * 4 / 3600:4.1f} h for 4 arms")
finally:
    recoverykit.release(student, teacher)
    student = teacher = None
    utils.free_gpu()
    utils.gpu_report("after probe")

11,401 blocks of 256 tokens = 2.92M tokens | 369 label-free MCQ prompts


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable LoRA parameters: 22.6M
step     0/20 | loss 0.8170 | kl_text 0.7995 | kl_mcq 0.0174 | 3.25s/step
step     5/20 | loss 1.1153 | kl_text 1.0096 | kl_mcq 0.1057 | 2.86s/step
step    10/20 | loss 1.0830 | kl_text 0.9517 | kl_mcq 0.1313 | 2.64s/step
step    15/20 | loss 1.0508 | kl_text 1.0018 | kl_mcq 0.0490 | 2.66s/step
step    19/20 | loss 0.7687 | kl_text 0.7281 | kl_mcq 0.0406 | 2.70s/step

median step time 2.68s (first 3 steps excluded)
peak GPU 0: 6.31 GiB
peak GPU 1: 6.63 GiB
  500 steps:    22 min per arm |  1.5 h for 4 arms
 1000 steps:    45 min per arm |  3.0 h for 4 arms
 1500 steps:    67 min per arm |  4.5 h for 4 arms
 2000 steps:    89 min per arm |  6.0 h for 4 arms
[gpu 0 after probe] free 14.38 / 14.56 GiB | allocated 0.02 | reserved 0.04
[gpu 1 after probe] free 14.41 / 14.56 GiB | allocated 0.01 | reserved 0.02


In [52]:
import json
import utils
import recoverykit

STEPS = 1000                    # same as your first run
SPEC_PATH = ROOT / "config" / "design_stage5_spec.json"
FORCE_SPEC = False

frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())
amend = json.loads((ROOT / "config" / "design_stage4_amendment.json").read_text())
spec = {**recoverykit.DEFAULT_SPEC, "steps": STEPS, "train_seed": frozen["config"]["seeds"]["train"],
        "arms": amend["amendment"]["arms_to_recover"],
        "parent_amendment_sha256": amend["amendment_sha256"],
        "data": {"text": "dom_text_train.jsonl", "mcq_prompts": "dom_mcq_train_nolabels.jsonl (labels removed)"},
        "identical_for_every_arm": True}
h = utils.sha256_json(spec)
if SPEC_PATH.exists() and not FORCE_SPEC:
    old = json.loads(SPEC_PATH.read_text())
    if old["spec_sha256"] != h:
        raise RuntimeError("The recovery spec is already frozen and differs from your edits. Revert, or set FORCE_SPEC=True deliberately.")
    print(f"[spec] already frozen at {old['frozen_at']}, unchanged")
else:
    SPEC_PATH.write_text(json.dumps({"frozen_at": utils.now_utc(), "spec_sha256": h, "spec": spec}, indent=2))
    print(f"[spec] wrote {SPEC_PATH.name} (sha {h[:12]}): {STEPS} steps for arms {spec['arms']}")

[spec] wrote design_stage5_spec.json (sha 386cf4debff3): 1000 steps for arms ['random', 'standard_importance', 'circuit_guided', 'standard_text']


In [53]:
import os
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError
import utils

def get_write_token():
    tok = os.environ.get("HF_WRITE_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
        except Exception:
            tok = None
    return tok

def backup_to_hf(root=ROOT, repo_name="specialist-backup"):
    tok = get_write_token()
    if not tok:
        raise RuntimeError("No HF_WRITE_TOKEN secret found. Add it in Kaggle (Add-ons > Secrets) and attach it to this notebook.")
    api = HfApi(token=tok)
    who = api.whoami()
    auth = who.get("auth", {}).get("accessToken", {})
    print(f"[backup] user: {who['name']} | token role: {auth.get('role')} | name: {auth.get('displayName')}")
    repo_id = f"{who['name']}/{repo_name}"
    try:
        api.repo_info(repo_id, repo_type="dataset")
    except RepositoryNotFoundError:
        api.create_repo(repo_id=repo_id, repo_type="dataset", private=True)
    api.upload_folder(folder_path=str(root), repo_id=repo_id, repo_type="dataset",
                      ignore_patterns=["artifacts/base_4bit_tmp/*", "**/ckpt.pt", "**/__pycache__/*"],
                      commit_message=f"backup {utils.now_utc()}")
    print(f"[backup] uploaded {root} to private dataset {repo_id}")

try:
    backup_to_hf()
except Exception as e:
    print(f"[backup] FAILED: {type(e).__name__}: {str(e)[:300]}")

# To restore after a reset (run Cell 0.1 first):
# from huggingface_hub import snapshot_download
# snapshot_download(f"{HfApi(token=get_write_token()).whoami()['name']}/specialist-backup", repo_type="dataset", local_dir=str(ROOT), token=get_write_token())

[backup] user: Anirudh-Peddinti | token role: write | name: kaggle-write
[backup] uploaded /kaggle/working/specialist to private dataset Anirudh-Peddinti/specialist-backup


circuit guided

In [54]:
import json, time, importlib, gc
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, recoverykit, utils
importlib.reload(recoverykit)

ONLY_ARMS = ["circuit_guided"]     # one arm per launch, e.g. ["standard_importance"] next; None = every unfinished arm

def host_ram(tag):
    rss = int([l for l in open("/proc/self/status") if l.startswith("VmRSS")][0].split()[1]) / 1024**2
    avail, total = utils.cpu_ram_gib()
    print(f"[host RAM {tag}] this process {rss:.1f} GiB | available {avail:.1f} of {total:.1f} GiB", flush=True)

def maybe_backup(tag):
    fn = globals().get("backup_to_hf")
    if fn is None:
        print(f"[backup] skipped after {tag}: run the backup cell once so backup_to_hf exists")
        return
    try:
        fn()
    except Exception as e:
        print(f"[backup] FAILED after {tag}: {type(e).__name__}: {str(e)[:200]}")

spec_rec = json.loads((ROOT / "config" / "design_stage5_spec.json").read_text())
spec, STEPS, ARMS = spec_rec["spec"], spec_rec["spec"]["steps"], spec_rec["spec"]["arms"]
frozen, rows, wiki_texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = datakit.read_jsonl(D / "dom_text_train.jsonl")["text"].tolist()
blocks = recoverykit.build_blocks(tokenizer, texts, spec["block"])
mcq_prompts = datakit.read_jsonl(D / "dom_mcq_train_nolabels.jsonl")["question"].tolist()
gen_val = datakit.read_jsonl(D / "gen_mcq_val.jsonl")
gen_prompts, gen_labels = gen_val["question"].tolist(), gen_val["answer_idx"].to_numpy()
dom_val_text = datakit.read_jsonl(D / "dom_text_val.jsonl")["text"].tolist()[:64]
K = torch.load(ROOT / "scores" / "keep_indices.pt")

todo = [a for a in ARMS if (ONLY_ARMS is None or a in ONLY_ARMS)
        and not (ROOT / "metrics" / f"stage5_recovery_{a}.json").exists()]
print("arms to run in this launch:", todo)

if not todo:
    print("nothing to do: every selected arm already has a result file")
else:
    utils.free_gpu()
    utils.assert_free_gpu(7, devices=[1])
    host_ram("before teacher")
    teacher = None
    try:
        teacher = scorekit.load_model(frozen["model_id"], 1)        # stays resident for the arms in this launch
        teacher.eval()
        host_ram("after teacher")
        for arm in todo:
            out_json = ROOT / "metrics" / f"stage5_recovery_{arm}.json"
            print(f"\n===== arm: {arm} ({STEPS} steps) =====")
            adir = ROOT / "artifacts" / "recovery" / arm
            adir.mkdir(parents=True, exist_ok=True)
            utils.assert_free_gpu(9, devices=[0])
            host_ram(f"{arm} start")
            torch.cuda.reset_peak_memory_stats(0)
            t0 = time.perf_counter()
            student = merged = None
            try:
                student = recoverykit.build_student(frozen["model_id"], K["keeps"][arm], 0, spec)
                log, lora_state = recoverykit.train(student, teacher, tokenizer, blocks, mcq_prompts, letter_ids, spec,
                                                    steps=STEPS, seed=spec["train_seed"], ckpt_path=adir / "ckpt.pt")
                torch.save(lora_state, adir / "adapter_final.pt")
                train_min = (time.perf_counter() - t0) / 60
                merged = student.merge_and_unload()                 # the deliverable is the merged plain model
                metrics = recoverykit.eval_all(merged, tokenizer, rows, gen_prompts, gen_labels,
                                               wiki_texts, dom_val_text, letter_ids)
            except BaseException:
                recoverykit.release(student, merged)
                raise
            metrics.update({"arm": arm, "steps": STEPS, "train_minutes": train_min, "log": log,
                            "peak_gpu0_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
                            "finished_at": utils.now_utc()})
            out_json.write_text(json.dumps(metrics))
            print(f"[{arm}] domain acc {metrics['dom_acc']:.3f} | off-domain acc {metrics['gen_acc']:.3f} | "
                  f"ppl wiki {metrics['ppl_wiki']:.2f} | ppl domain {metrics['ppl_dom']:.2f} | {train_min:.0f} min")
            del student, merged, lora_state, log, metrics
            gc.collect()
            utils.free_gpu()
            host_ram(f"{arm} end")
            maybe_backup(arm)
    finally:
        recoverykit.release(teacher)
        teacher = None
        utils.free_gpu()
        utils.gpu_report("after recovery")
    print("done for:", todo)

arms to run in this launch: ['circuit_guided']
[host RAM before teacher] this process 4.2 GiB | available 25.8 of 31.3 GiB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[host RAM after teacher] this process 4.2 GiB | available 25.3 of 31.3 GiB

===== arm: circuit_guided (1000 steps) =====
[host RAM circuit_guided start] this process 4.2 GiB | available 25.3 of 31.3 GiB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

step     0/1000 | loss 0.8170 | kl_text 0.7995 | kl_mcq 0.0174 | 2.85s/step
step    10/1000 | loss 1.0854 | kl_text 0.9522 | kl_mcq 0.1331 | 2.69s/step
step    20/1000 | loss 0.9971 | kl_text 0.7596 | kl_mcq 0.2375 | 2.85s/step
step    30/1000 | loss 0.7120 | kl_text 0.6702 | kl_mcq 0.0418 | 2.65s/step
step    40/1000 | loss 0.6437 | kl_text 0.5991 | kl_mcq 0.0446 | 2.67s/step
step    50/1000 | loss 0.7872 | kl_text 0.6656 | kl_mcq 0.1216 | 2.81s/step
step    60/1000 | loss 0.6372 | kl_text 0.5736 | kl_mcq 0.0635 | 2.65s/step
step    70/1000 | loss 0.6594 | kl_text 0.5919 | kl_mcq 0.0675 | 2.66s/step
step    80/1000 | loss 0.7050 | kl_text 0.6607 | kl_mcq 0.0443 | 2.66s/step
step    90/1000 | loss 0.7460 | kl_text 0.6082 | kl_mcq 0.1378 | 2.86s/step
step   100/1000 | loss 0.6756 | kl_text 0.5107 | kl_mcq 0.1650 | 2.81s/step
step   110/1000 | loss 0.7883 | kl_text 0.6829 | kl_mcq 0.1054 | 2.79s/step
step   120/1000 | loss 0.5251 | kl_text 0.4680 | kl_mcq 0.0571 | 2.66s/step
step   130/1

standard_importance

In [55]:
import json, time, importlib, gc
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, recoverykit, utils
importlib.reload(recoverykit)

ONLY_ARMS = ["standard_importance"]     # one arm per launch, e.g. ["standard_importance"] next; None = every unfinished arm

def host_ram(tag):
    rss = int([l for l in open("/proc/self/status") if l.startswith("VmRSS")][0].split()[1]) / 1024**2
    avail, total = utils.cpu_ram_gib()
    print(f"[host RAM {tag}] this process {rss:.1f} GiB | available {avail:.1f} of {total:.1f} GiB", flush=True)

def maybe_backup(tag):
    fn = globals().get("backup_to_hf")
    if fn is None:
        print(f"[backup] skipped after {tag}: run the backup cell once so backup_to_hf exists")
        return
    try:
        fn()
    except Exception as e:
        print(f"[backup] FAILED after {tag}: {type(e).__name__}: {str(e)[:200]}")

spec_rec = json.loads((ROOT / "config" / "design_stage5_spec.json").read_text())
spec, STEPS, ARMS = spec_rec["spec"], spec_rec["spec"]["steps"], spec_rec["spec"]["arms"]
frozen, rows, wiki_texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = datakit.read_jsonl(D / "dom_text_train.jsonl")["text"].tolist()
blocks = recoverykit.build_blocks(tokenizer, texts, spec["block"])
mcq_prompts = datakit.read_jsonl(D / "dom_mcq_train_nolabels.jsonl")["question"].tolist()
gen_val = datakit.read_jsonl(D / "gen_mcq_val.jsonl")
gen_prompts, gen_labels = gen_val["question"].tolist(), gen_val["answer_idx"].to_numpy()
dom_val_text = datakit.read_jsonl(D / "dom_text_val.jsonl")["text"].tolist()[:64]
K = torch.load(ROOT / "scores" / "keep_indices.pt")

todo = [a for a in ARMS if (ONLY_ARMS is None or a in ONLY_ARMS)
        and not (ROOT / "metrics" / f"stage5_recovery_{a}.json").exists()]
print("arms to run in this launch:", todo)

if not todo:
    print("nothing to do: every selected arm already has a result file")
else:
    utils.free_gpu()
    utils.assert_free_gpu(7, devices=[1])
    host_ram("before teacher")
    teacher = None
    try:
        teacher = scorekit.load_model(frozen["model_id"], 1)        # stays resident for the arms in this launch
        teacher.eval()
        host_ram("after teacher")
        for arm in todo:
            out_json = ROOT / "metrics" / f"stage5_recovery_{arm}.json"
            print(f"\n===== arm: {arm} ({STEPS} steps) =====")
            adir = ROOT / "artifacts" / "recovery" / arm
            adir.mkdir(parents=True, exist_ok=True)
            utils.assert_free_gpu(9, devices=[0])
            host_ram(f"{arm} start")
            torch.cuda.reset_peak_memory_stats(0)
            t0 = time.perf_counter()
            student = merged = None
            try:
                student = recoverykit.build_student(frozen["model_id"], K["keeps"][arm], 0, spec)
                log, lora_state = recoverykit.train(student, teacher, tokenizer, blocks, mcq_prompts, letter_ids, spec,
                                                    steps=STEPS, seed=spec["train_seed"], ckpt_path=adir / "ckpt.pt")
                torch.save(lora_state, adir / "adapter_final.pt")
                train_min = (time.perf_counter() - t0) / 60
                merged = student.merge_and_unload()                 # the deliverable is the merged plain model
                metrics = recoverykit.eval_all(merged, tokenizer, rows, gen_prompts, gen_labels,
                                               wiki_texts, dom_val_text, letter_ids)
            except BaseException:
                recoverykit.release(student, merged)
                raise
            metrics.update({"arm": arm, "steps": STEPS, "train_minutes": train_min, "log": log,
                            "peak_gpu0_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
                            "finished_at": utils.now_utc()})
            out_json.write_text(json.dumps(metrics))
            print(f"[{arm}] domain acc {metrics['dom_acc']:.3f} | off-domain acc {metrics['gen_acc']:.3f} | "
                  f"ppl wiki {metrics['ppl_wiki']:.2f} | ppl domain {metrics['ppl_dom']:.2f} | {train_min:.0f} min")
            del student, merged, lora_state, log, metrics
            gc.collect()
            utils.free_gpu()
            host_ram(f"{arm} end")
            maybe_backup(arm)
    finally:
        recoverykit.release(teacher)
        teacher = None
        utils.free_gpu()
        utils.gpu_report("after recovery")
    print("done for:", todo)

arms to run in this launch: ['standard_importance']
[host RAM before teacher] this process 4.2 GiB | available 26.3 of 31.3 GiB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[host RAM after teacher] this process 4.3 GiB | available 26.0 of 31.3 GiB

===== arm: standard_importance (1000 steps) =====
[host RAM standard_importance start] this process 4.3 GiB | available 26.0 of 31.3 GiB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

step     0/1000 | loss 0.7877 | kl_text 0.6283 | kl_mcq 0.1593 | 2.83s/step
step    10/1000 | loss 0.6954 | kl_text 0.6072 | kl_mcq 0.0882 | 2.66s/step
step    20/1000 | loss 0.8937 | kl_text 0.6126 | kl_mcq 0.2811 | 2.86s/step
step    30/1000 | loss 0.6178 | kl_text 0.5215 | kl_mcq 0.0963 | 2.67s/step
step    40/1000 | loss 0.5527 | kl_text 0.4736 | kl_mcq 0.0791 | 2.66s/step
step    50/1000 | loss 0.4893 | kl_text 0.4741 | kl_mcq 0.0152 | 2.82s/step
step    60/1000 | loss 0.4917 | kl_text 0.4034 | kl_mcq 0.0883 | 2.64s/step
step    70/1000 | loss 0.4931 | kl_text 0.4060 | kl_mcq 0.0870 | 2.69s/step
step    80/1000 | loss 0.7873 | kl_text 0.4516 | kl_mcq 0.3358 | 2.67s/step
step    90/1000 | loss 0.7948 | kl_text 0.4681 | kl_mcq 0.3268 | 2.82s/step
step   100/1000 | loss 0.6351 | kl_text 0.3882 | kl_mcq 0.2469 | 2.86s/step
step   110/1000 | loss 0.6050 | kl_text 0.5098 | kl_mcq 0.0953 | 2.80s/step
step   120/1000 | loss 0.4313 | kl_text 0.3588 | kl_mcq 0.0725 | 2.67s/step
step   130/1

random

In [56]:
import json, time, importlib, gc
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, recoverykit, utils
importlib.reload(recoverykit)

ONLY_ARMS = ["random"]     # one arm per launch, e.g. ["standard_importance"] next; None = every unfinished arm

def host_ram(tag):
    rss = int([l for l in open("/proc/self/status") if l.startswith("VmRSS")][0].split()[1]) / 1024**2
    avail, total = utils.cpu_ram_gib()
    print(f"[host RAM {tag}] this process {rss:.1f} GiB | available {avail:.1f} of {total:.1f} GiB", flush=True)

def maybe_backup(tag):
    fn = globals().get("backup_to_hf")
    if fn is None:
        print(f"[backup] skipped after {tag}: run the backup cell once so backup_to_hf exists")
        return
    try:
        fn()
    except Exception as e:
        print(f"[backup] FAILED after {tag}: {type(e).__name__}: {str(e)[:200]}")

spec_rec = json.loads((ROOT / "config" / "design_stage5_spec.json").read_text())
spec, STEPS, ARMS = spec_rec["spec"], spec_rec["spec"]["steps"], spec_rec["spec"]["arms"]
frozen, rows, wiki_texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = datakit.read_jsonl(D / "dom_text_train.jsonl")["text"].tolist()
blocks = recoverykit.build_blocks(tokenizer, texts, spec["block"])
mcq_prompts = datakit.read_jsonl(D / "dom_mcq_train_nolabels.jsonl")["question"].tolist()
gen_val = datakit.read_jsonl(D / "gen_mcq_val.jsonl")
gen_prompts, gen_labels = gen_val["question"].tolist(), gen_val["answer_idx"].to_numpy()
dom_val_text = datakit.read_jsonl(D / "dom_text_val.jsonl")["text"].tolist()[:64]
K = torch.load(ROOT / "scores" / "keep_indices.pt")

todo = [a for a in ARMS if (ONLY_ARMS is None or a in ONLY_ARMS)
        and not (ROOT / "metrics" / f"stage5_recovery_{a}.json").exists()]
print("arms to run in this launch:", todo)

if not todo:
    print("nothing to do: every selected arm already has a result file")
else:
    utils.free_gpu()
    utils.assert_free_gpu(7, devices=[1])
    host_ram("before teacher")
    teacher = None
    try:
        teacher = scorekit.load_model(frozen["model_id"], 1)        # stays resident for the arms in this launch
        teacher.eval()
        host_ram("after teacher")
        for arm in todo:
            out_json = ROOT / "metrics" / f"stage5_recovery_{arm}.json"
            print(f"\n===== arm: {arm} ({STEPS} steps) =====")
            adir = ROOT / "artifacts" / "recovery" / arm
            adir.mkdir(parents=True, exist_ok=True)
            utils.assert_free_gpu(9, devices=[0])
            host_ram(f"{arm} start")
            torch.cuda.reset_peak_memory_stats(0)
            t0 = time.perf_counter()
            student = merged = None
            try:
                student = recoverykit.build_student(frozen["model_id"], K["keeps"][arm], 0, spec)
                log, lora_state = recoverykit.train(student, teacher, tokenizer, blocks, mcq_prompts, letter_ids, spec,
                                                    steps=STEPS, seed=spec["train_seed"], ckpt_path=adir / "ckpt.pt")
                torch.save(lora_state, adir / "adapter_final.pt")
                train_min = (time.perf_counter() - t0) / 60
                merged = student.merge_and_unload()                 # the deliverable is the merged plain model
                metrics = recoverykit.eval_all(merged, tokenizer, rows, gen_prompts, gen_labels,
                                               wiki_texts, dom_val_text, letter_ids)
            except BaseException:
                recoverykit.release(student, merged)
                raise
            metrics.update({"arm": arm, "steps": STEPS, "train_minutes": train_min, "log": log,
                            "peak_gpu0_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
                            "finished_at": utils.now_utc()})
            out_json.write_text(json.dumps(metrics))
            print(f"[{arm}] domain acc {metrics['dom_acc']:.3f} | off-domain acc {metrics['gen_acc']:.3f} | "
                  f"ppl wiki {metrics['ppl_wiki']:.2f} | ppl domain {metrics['ppl_dom']:.2f} | {train_min:.0f} min")
            del student, merged, lora_state, log, metrics
            gc.collect()
            utils.free_gpu()
            host_ram(f"{arm} end")
            maybe_backup(arm)
    finally:
        recoverykit.release(teacher)
        teacher = None
        utils.free_gpu()
        utils.gpu_report("after recovery")
    print("done for:", todo)

arms to run in this launch: ['random']
[host RAM before teacher] this process 4.3 GiB | available 26.3 of 31.3 GiB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[host RAM after teacher] this process 4.3 GiB | available 26.0 of 31.3 GiB

===== arm: random (1000 steps) =====
[host RAM random start] this process 4.3 GiB | available 26.0 of 31.3 GiB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

step     0/1000 | loss 8.5622 | kl_text 6.2978 | kl_mcq 2.2644 | 2.79s/step
step    10/1000 | loss 6.5732 | kl_text 5.4252 | kl_mcq 1.1480 | 2.70s/step
step    20/1000 | loss 3.6958 | kl_text 3.6350 | kl_mcq 0.0608 | 2.81s/step
step    30/1000 | loss 3.2142 | kl_text 2.6175 | kl_mcq 0.5967 | 2.64s/step
step    40/1000 | loss 2.7228 | kl_text 1.9474 | kl_mcq 0.7754 | 2.68s/step
step    50/1000 | loss 2.8197 | kl_text 1.6742 | kl_mcq 1.1455 | 2.81s/step
step    60/1000 | loss 1.4911 | kl_text 1.3018 | kl_mcq 0.1892 | 2.65s/step
step    70/1000 | loss 1.4896 | kl_text 1.3329 | kl_mcq 0.1567 | 2.64s/step
step    80/1000 | loss 1.4515 | kl_text 1.1585 | kl_mcq 0.2930 | 2.65s/step
step    90/1000 | loss 1.3431 | kl_text 1.2334 | kl_mcq 0.1097 | 2.82s/step
step   100/1000 | loss 1.5224 | kl_text 0.8942 | kl_mcq 0.6281 | 2.83s/step
step   110/1000 | loss 1.2230 | kl_text 1.1319 | kl_mcq 0.0910 | 2.82s/step
step   120/1000 | loss 1.6594 | kl_text 0.8319 | kl_mcq 0.8275 | 2.65s/step
step   130/1

standard_text

In [60]:
import json, time, importlib, gc
import numpy as np, torch
from transformers import AutoTokenizer
import evalkit, scorekit, datakit, recoverykit, utils
importlib.reload(recoverykit)

ONLY_ARMS = ["standard_text"]     # one arm per launch, e.g. ["standard_importance"] next; None = every unfinished arm

def host_ram(tag):
    rss = int([l for l in open("/proc/self/status") if l.startswith("VmRSS")][0].split()[1]) / 1024**2
    avail, total = utils.cpu_ram_gib()
    print(f"[host RAM {tag}] this process {rss:.1f} GiB | available {avail:.1f} of {total:.1f} GiB", flush=True)

def maybe_backup(tag):
    fn = globals().get("backup_to_hf")
    if fn is None:
        print(f"[backup] skipped after {tag}: run the backup cell once so backup_to_hf exists")
        return
    try:
        fn()
    except Exception as e:
        print(f"[backup] FAILED after {tag}: {type(e).__name__}: {str(e)[:200]}")

spec_rec = json.loads((ROOT / "config" / "design_stage5_spec.json").read_text())
spec, STEPS, ARMS = spec_rec["spec"], spec_rec["spec"]["steps"], spec_rec["spec"]["arms"]
frozen, rows, wiki_texts, letter_ids = evalkit.load_eval_inputs(ROOT)
D = ROOT / "data"
tokenizer = AutoTokenizer.from_pretrained(frozen["model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = datakit.read_jsonl(D / "dom_text_train.jsonl")["text"].tolist()
blocks = recoverykit.build_blocks(tokenizer, texts, spec["block"])
mcq_prompts = datakit.read_jsonl(D / "dom_mcq_train_nolabels.jsonl")["question"].tolist()
gen_val = datakit.read_jsonl(D / "gen_mcq_val.jsonl")
gen_prompts, gen_labels = gen_val["question"].tolist(), gen_val["answer_idx"].to_numpy()
dom_val_text = datakit.read_jsonl(D / "dom_text_val.jsonl")["text"].tolist()[:64]
K = torch.load(ROOT / "scores" / "keep_indices.pt")

todo = [a for a in ARMS if (ONLY_ARMS is None or a in ONLY_ARMS)
        and not (ROOT / "metrics" / f"stage5_recovery_{a}.json").exists()]
print("arms to run in this launch:", todo)

if not todo:
    print("nothing to do: every selected arm already has a result file")
else:
    utils.free_gpu()
    utils.assert_free_gpu(7, devices=[1])
    host_ram("before teacher")
    teacher = None
    try:
        teacher = scorekit.load_model(frozen["model_id"], 1)        # stays resident for the arms in this launch
        teacher.eval()
        host_ram("after teacher")
        for arm in todo:
            out_json = ROOT / "metrics" / f"stage5_recovery_{arm}.json"
            print(f"\n===== arm: {arm} ({STEPS} steps) =====")
            adir = ROOT / "artifacts" / "recovery" / arm
            adir.mkdir(parents=True, exist_ok=True)
            utils.assert_free_gpu(9, devices=[0])
            host_ram(f"{arm} start")
            torch.cuda.reset_peak_memory_stats(0)
            t0 = time.perf_counter()
            student = merged = None
            try:
                student = recoverykit.build_student(frozen["model_id"], K["keeps"][arm], 0, spec)
                log, lora_state = recoverykit.train(student, teacher, tokenizer, blocks, mcq_prompts, letter_ids, spec,
                                                    steps=STEPS, seed=spec["train_seed"], ckpt_path=adir / "ckpt.pt")
                torch.save(lora_state, adir / "adapter_final.pt")
                train_min = (time.perf_counter() - t0) / 60
                merged = student.merge_and_unload()                 # the deliverable is the merged plain model
                metrics = recoverykit.eval_all(merged, tokenizer, rows, gen_prompts, gen_labels,
                                               wiki_texts, dom_val_text, letter_ids)
            except BaseException:
                recoverykit.release(student, merged)
                raise
            metrics.update({"arm": arm, "steps": STEPS, "train_minutes": train_min, "log": log,
                            "peak_gpu0_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
                            "finished_at": utils.now_utc()})
            out_json.write_text(json.dumps(metrics))
            print(f"[{arm}] domain acc {metrics['dom_acc']:.3f} | off-domain acc {metrics['gen_acc']:.3f} | "
                  f"ppl wiki {metrics['ppl_wiki']:.2f} | ppl domain {metrics['ppl_dom']:.2f} | {train_min:.0f} min")
            del student, merged, lora_state, log, metrics
            gc.collect()
            utils.free_gpu()
            host_ram(f"{arm} end")
            maybe_backup(arm)
    finally:
        recoverykit.release(teacher)
        teacher = None
        utils.free_gpu()
        utils.gpu_report("after recovery")
    print("done for:", todo)

arms to run in this launch: []
nothing to do: every selected arm already has a result file


In [95]:
import json
import numpy as np, pandas as pd
import utils

spec = json.loads((ROOT / "config" / "design_stage5_spec.json").read_text())["spec"]
chosen = json.loads((ROOT / "config" / "stage3_budget_choice.json").read_text())["chosen_mlp_neuron_reduction"]
sweep = json.loads((ROOT / "metrics" / "stage3_sweep.json").read_text())
off = json.loads((ROOT / "metrics" / "stage3_sweep_offdomain.json").read_text())
frozen = json.loads((ROOT / "config" / "criteria_frozen.json").read_text())

arms_done = [a for a in spec["arms"] if (ROOT / "metrics" / f"stage5_recovery_{a}.json").exists()]
assert arms_done, "no recovery results yet: run Cell 5.4"
print("arms with results:", arms_done, "| still missing:", [a for a in spec["arms"] if a not in arms_done])
post = {a: json.loads((ROOT / "metrics" / f"stage5_recovery_{a}.json").read_text()) for a in arms_done}

CH = 0.25
b = {"dom": sweep["base"]["acc"], "gen": off["base"]["gen_acc"], "ppl_w": sweep["base"]["ppl"], "ppl_d": off["base"]["dom_text_ppl"]}
skill = lambda acc, base: (acc - CH) / (base - CH)

def pre_key(arm):
    return f"random|{chosen}|0" if arm == "random" else f"{arm}|{chosen}"

rows = [{"arm": "unpruned", "stage": "-", "dom_acc": b["dom"], "gen_acc": b["gen"], "dom_skill": 1.0, "gen_skill": 1.0,
         "ppl_dom_x": 1.0, "ppl_wiki_x": 1.0}]
for a in arms_done:
    k = pre_key(a)
    rows.append({"arm": a, "stage": "pre", "dom_acc": sweep[k]["acc"], "gen_acc": off[k]["gen_acc"],
                 "dom_skill": skill(sweep[k]["acc"], b["dom"]), "gen_skill": skill(off[k]["gen_acc"], b["gen"]),
                 "ppl_dom_x": off[k]["dom_text_ppl"] / b["ppl_d"], "ppl_wiki_x": sweep[k]["ppl"] / b["ppl_w"]})
    p = post[a]
    rows.append({"arm": a, "stage": "post", "dom_acc": p["dom_acc"], "gen_acc": p["gen_acc"],
                 "dom_skill": skill(p["dom_acc"], b["dom"]), "gen_skill": skill(p["gen_acc"], b["gen"]),
                 "ppl_dom_x": p["ppl_dom"] / b["ppl_d"], "ppl_wiki_x": p["ppl_wiki"] / b["ppl_w"]})
print(f"{spec['steps']} recovery steps per arm; skill = (accuracy - 0.25) / (unpruned accuracy - 0.25)")
print(pd.DataFrame(rows).round(3).to_string(index=False))

def gap_diff_ci(a, b_, n_boot=5000, seed=0):
    (ad, ag), (bd, bg) = a, b_
    rng = np.random.default_rng(seed)
    di = rng.integers(0, len(ad), (n_boot, len(ad)))
    gi = rng.integers(0, len(ag), (n_boot, len(ag)))
    stat = (ad[di].mean(1) - ag[gi].mean(1)) - (bd[di].mean(1) - bg[gi].mean(1))
    return float(((ad.mean() - ag.mean()) - (bd.mean() - bg.mean())) * 100), *[float(x) * 100 for x in np.percentile(stat, [2.5, 97.5])]

arr = {a: (np.array(post[a]["dom_correct"], float), np.array(post[a]["gen_correct"], float)) for a in arms_done}
base_arr = (np.array(sweep["base"]["correct"], float), np.array(off["base"]["gen_correct"], float))
print("\nAfter recovery: change in the domain-minus-general gap vs unpruned, in points (95% CI)")
for a in arms_done:
    print("  {:22s} {:+6.1f} ({:+.1f} to {:+.1f})".format(a, *gap_diff_ci(arr[a], base_arr)))

bseed = frozen["config"]["seeds"]["bootstrap"]
if "circuit_guided" in arr:
    print("\ncircuit_guided against each other arm (gap in points; domain accuracy in points; 95% CI):")
    for other in [a for a in ("standard_importance", "random", "standard_text") if a in arr]:
        g = gap_diff_ci(arr["circuit_guided"], arr[other])
        d = utils.paired_bootstrap_diff(arr["circuit_guided"][0], arr[other][0], seed=bseed)
        tag = "reference arm" if other == "standard_text" else "claim control"
        print("  minus {:20s} ({}): gap {:+6.1f} ({:+.1f} to {:+.1f}) | domain accuracy {:+.1f} ({:+.1f} to {:+.1f})".format(
            other, tag, *g, d[0] * 100, d[1][0] * 100, d[1][1] * 100))
else:
    print("\ncircuit_guided has no result yet, so no head-to-head comparison.")
print("Development sets only (dev_eval n=250, gen_mcq_val n=200). The sealed test is scored once, in Stage 8.")

arms with results: ['random', 'standard_importance', 'circuit_guided', 'standard_text'] | still missing: []
1000 recovery steps per arm; skill = (accuracy - 0.25) / (unpruned accuracy - 0.25)
                arm stage  dom_acc  gen_acc  dom_skill  gen_skill  ppl_dom_x  ppl_wiki_x
           unpruned     -    0.596    0.590      1.000      1.000      1.000       1.000
             random   pre    0.224    0.220     -0.075     -0.088    991.907     559.632
             random  post    0.384    0.315      0.387      0.191      1.638       3.461
standard_importance   pre    0.528    0.385      0.803      0.397      1.807       3.663
standard_importance  post    0.576    0.420      0.942      0.500      1.283       2.321
     circuit_guided   pre    0.520    0.375      0.780      0.368      2.366       5.292
     circuit_guided  post    0.568    0.420      0.919      0.500      1.385       2.628
      standard_text   pre    0.512    0.330      0.757      0.235      1.526       4.061
      s